# Automação — Energia Injetada (Geração Distribuída Copel) · v2 com rateio anual

**Pipeline completo em 5 etapas**, do PDF ao relatório:

| Etapa | O quê | Biblioteca |
|------|-------|-----------|
| **1** | Ler o *Demonstrativo de Micro/Minigeração* (PDF) e extrair, por UC, os dados do mês mais recente (+ cadastro: fase e custo de disponibilidade) | `pdfplumber` + `re` |
| **2** | Atualizar `Acompanhamento Energia Injetada PR.xlsx` preservando fórmulas e formatação | `openpyxl` |
| **3** | Gerar o **painel executivo** (PNG/PDF) com status de saldo e recomendações | `matplotlib` |
| **4** | **Rateio anual**: calcular o % de cada loja para o próximo ciclo de 12 meses (cobertura equalizada), simular mês a mês e comparar métodos sob estresse | `numpy` + `scipy` |
| **5** | **Saídas**: abas `Rateio *` no `.xlsx` e **relatório PDF** (painel + rateio + gráficos + premissas) | `openpyxl` + `matplotlib` |

> **O que mudou em relação à v1:** as Etapas 1–3 são as mesmas. A `% Ideal` do painel agora vem do
> **modelo de rateio anual** (Etapa 4), e não mais da regra `(12 + 0,5) × consumo médio − saldo`.
> A regra antiga continua disponível (`ESTRATEGIA_RATEIO = "C"`) e aparece na comparação de métodos.

---

### Arquitetura da planilha (mapeada a partir do arquivo real)

A planilha **não guarda dados nas abas de loja** — elas são *derivadas por fórmula*. Toda a
informação bruta vive em **uma única aba-mestre `Demonstrativo`**:

```
Demonstrativo (aba-mestre, 1 linha por UC × mês)
 ├─ A nome_loja   B uc_conces   C consumo   D saldo_anterior_fp
 ├─ E cred_receb_fp   F percentual   G saldo_final_fp   H arquivo   I data_referencia
 │
 ├──► Abas de loja ("01 - MATRIZ", …)  → SUMIFS por loja/mês
 ├──► "Resumo Ultimo Mês"              → SUMIFS com P1 = =MAX(Demonstrativo!I:I)  (auto!)
 ├──► "Resumo Anual (2026)"            → SUMIFS do ano  (auto!)
 └──► "Rateio Anual", "Rateio Simulação", "Rateio Métodos", "Rateio Premissas"  (Etapa 5, v2)
```

> **Consequência prática:** basta **inserir as novas linhas na aba `Demonstrativo`**. Os dois
> resumos se atualizam sozinhos (a célula de referência é `=MAX(...)`). As abas de loja recebem,
> opcionalmente, uma nova linha de fórmula por mês. As abas `Rateio *` são **recriadas** a cada execução.

### Mapeamento coluna do PDF → coluna do `Demonstrativo` (validado nos dados)

| Coluna do PDF (soma das sub-colunas) | Coluna no `Demonstrativo` |
|---|---|
| Energia Ativa            | `consumo` |
| Saldo Anterior           | `saldo_anterior_fp` |
| Cred. Receb. Outra UC    | `cred_receb_fp`  *(exibida como "Energia Injetada" nas abas de loja)* |
| Percentual (%)           | `percentual` |
| Saldo Final              | `saldo_final_fp` |

**Particularidades do PDF** (já tratadas no código):
- A ordem das sub-colunas **inverte**: página **geradora** = `PT | FP`; páginas **beneficiárias** = `FP | TP`.
  → resolvido por `somar_energia()`, que soma as duas sub-colunas (são mutuamente exclusivas).
- O `UC Conces` deve ser lido do **cabeçalho** (`top < 100`) para não pegar a UC geradora que se repete.
- `limpar_numero()` só trata ponto como separador de milhar **quando há vírgula** na string.
- (v2) O cabeçalho traz `Tipo da Fase` e `Valor do Custo de Disp. Kwh` — a parte do consumo que **nunca**
  é compensada (TR = 100, BI = 50, MO = 30 kWh/mês). O rateio usa o consumo **compensável**.

> **Padrão da planilha (v3):** a planilha-base `Acompanhamento Energia Injetada PR.xlsx` define o visual (cabeçalhos cinza, abas laranja/azul, aba `Unidades Consumidoras`). O notebook só acrescenta dados e fórmulas, preserva os meses já montados nas abas das lojas (`MANTER_MESES_EXISTENTES`) e copia o estilo da linha anterior nas linhas novas do `Demonstrativo`. As abas `Rateio *` seguem o mesmo padrão.

## 1. Configuração e logging
Todos os caminhos, o mapeamento de colunas e as **regras de negócio** ficam aqui — ajuste sem tocar no restante.

In [ ]:
from __future__ import annotations
import logging, sys
from pathlib import Path

# ---------------------------------------------------------------- CAMINHOS
BASE_DIR   = Path.cwd()                                                # pasta do notebook
PDF_PATH   = BASE_DIR / "demonstativos" / "demonstrativo-CD.pdf"     # entrada: demonstrativo
XLSX_PATH  = BASE_DIR / "Acompanhamento Energia Injetada PR.xlsx"       # planilha a atualizar
OUT_DIR    = BASE_DIR / "saida"; OUT_DIR.mkdir(exist_ok=True)           # onde salvar relatórios
XLSX_OUT   = OUT_DIR / "Acompanhamento Energia Injetada PR.xlsx"
PAINEL_OUT = OUT_DIR / "painel_energia_injetada PR.pdf"                     # sem extensão (gera .png e .pdf)
RELATORIO_OUT = OUT_DIR / "relatorio_energia_injetada PR.pdf"           # (v2) relatório completo

# ---------------------------------------------------------------- ESTRUTURA DA ABA-MESTRE
ABA_MESTRE   = "Demonstrativo"
COLS_MESTRE  = ["nome_loja", "uc_conces", "consumo", "saldo_anterior_fp",
                "cred_receb_fp", "percentual", "saldo_final_fp", "arquivo", "data_referencia"]
COL_IDX = {c: i + 1 for i, c in enumerate(COLS_MESTRE)}          # 1-based p/ openpyxl
ABAS_RESUMO  = ["Resumo Ultimo Mês", "Resumo Anual (2026)"]
TAG_GERADORA = "GERADORA"                                        # identifica a UC geradora pelo nome

# ---------------------------------------------------------------- UCs (v4)
# O demonstrativo Copel novo não traz mais o "UC Conces."; a chave da UC passa a ser o nº da UC
# (15 dígitos, ex.: 000.097.375.703.120). O nome de UCs NOVAS vem da aba "Unidades Consumidoras"
# da planilha (colunas uc_conces | nome_loja, com ou sem pontos); UCs já gravadas mantêm o nome do
# Demonstrativo (é o critério dos SUMIFS das abas de loja).
ABA_UCS = "Unidades Consumidoras"
CRIAR_ABAS_NOVAS_LOJAS = True    # loja nova (UC fora do Demonstrativo): cria aba + linha nos resumos
# Migração única do histórico: UC Conces. antigo -> UC nova (pares lidos do demonstrativo antigo,
# que trazia os dois números). Linhas do Demonstrativo com o nº antigo são convertidas ao carregar.
MIGRACAO_UC_CONCES: dict[int, int] = {
    83331379: 97375703120,     # 01 - MATRIZ
    88896455: 132992503112,    # 02 - BOM RETIRO
    83237992: 223868603167,    # 03 - BACACHERI
    54145163: 219794603178,    # 04 - CENTRO CIVICO
    57429200: 209871203131,    # 05 - AGUA VERDE
    57429294: 213163603169,    # 05 - AGUA VERDE (2ª UC)
    65590198: 337979803167,    # 06 - SEMINÁRIO
    87260174: 248327503143,    # 10 - COLOMBO
    75883252: 81203403145,     # 11 - PORTÃO
    56497253: 380247903172,    # 12 - ARAUCARIA
    76735273: 155897503114,    # 13 - ALTO DA XV
    90820207: 283899703122,    # 14 - CAPÃO RASO
    72999551: 207602203175,    # 15 - JD DAS AMÉRICAS
    74462253: 223258303135,    # 16 - SÃO BRAZ
    946389:   127210003104,    # 17 - XAXIM
    73154610: 26432703184,     # 18 - HAUER
    57360332: 334120603100,    # 19 - FAZENDA
    6630006:  238930303100,    # 20 - PINHAIS
    77083776: 313986003158,    # 21 - SÃO JOSÉ
    92705154: 480291403129,    # 22 - TARUMÃ
    66227682: 228536003117,    # 23 - LINDÓIA
    7396538:  291665103144,    # 24 - AHÚ
    3858766:  43513903197,     # 25 - PONTA GROSSA-PG
    33019479: 406469403155,    # 26 - AFONSO PENA
    1102478:  221034503101,    # 27 - CAMPINA DO SIQ.
    104810009: 582100903121,   # 28 - IRAI
    45309809: 62792103120,     # 29 - CAMPINA GRANDE
    3954285:  50325103150,     # 33 - ESTRELA-PG
    108591239: 616202103162,   # 35 - UMBARÁ
    23909480: 312427903146,    # 37 - CAJURU
    1267558:  326075703175,    # 38 - MARECHAL
    43756174: 365308203178,    # 39 - JOÃO BETTGA
    82826714: 244474403114,    # 40 - MARINGÁ
    86020463: 349435003150,    # 41 - UVARANAS
    19987161: 348708803147,    # 42 - MATINHOS
    96550937: 512805403104,    # 80 - INDUSTRIA (GERADORA)
    95562036: 504133603169,
    97901768: 525550103106,
}
FMT_UC = "000000000000000"       # UC exibida com 15 dígitos, como no PDF

# ---------------------------------------------------------------- OPÇÕES DE EXECUÇÃO
DEDUPLICAR_UCS      = False   # o arquivo real mantém 1 linha "real" + 1 linha zero por UC; False preserva o padrão
ATUALIZAR_ABAS_LOJA = True    # inserir nova linha de fórmula por mês nas abas individuais
MANTER_MESES_EXISTENTES = True  # mantém os meses já montados nas abas de loja (ex.: calendário até dezembro)
TENTAR_OCR          = True    # se o PDF não tiver camada de texto, tentar OCR (requer pytesseract+poppler)

# ---------------------------------------------------------------- REGRAS DE NEGÓCIO (Etapa 3)
LIM_SUPERAVIT      = 2000     # saldo_final > X kWh            -> "Superávit"
LIM_CONFORTAVEL    = 800      # LIM_CONFORTAVEL < saldo <= LIM_SUPERAVIT -> "Confortável"
LIM_BAIXO          = 1        # LIM_BAIXO <= saldo <= LIM_CONFORTAVEL    -> "Saldo baixo"
                              #  saldo < LIM_BAIXO             -> "Crítico / Zero"

# ---------------------------------------------------------------- % IDEAL (Etapa 3)
# A % Ideal vem do rateio anual (Etapa 4) — parâmetros na célula 4.0.

# ---- parâmetros das estratégias legadas "C" e "A" (mantidos da v1)
HORIZONTE_MESES    = 12       # meses até a próxima revisão do rateio: 12 (1x/ano) ou 6 (2x/ano)
CONSIDERAR_SALDO   = True     # legado: True = estratégia C; False = estratégia A
JANELA_MEDIA_MESES = 12       # nº de meses mais recentes na média de consumo (None = todo o histórico)
MESES_BUFFER       = 0.5      # margem de segurança da estratégia C
PISO_PCT           = 0.0      # % mínima por filial (0 = sem piso); o excedente é renormalizado

EMPRESA = "Comercio de Tintas Material Eletrico e Hidraulico Verginia Ltda"
UF      = "Paraná"

# ---------------------------------------------------------------- LOGGING
logger = logging.getLogger("energia")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_h = logging.StreamHandler(sys.stdout)
_h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
logger.addHandler(_h)
logger.info("Configuração carregada. PDF=%s | XLSX=%s", PDF_PATH.name, XLSX_PATH.name)

## 2. Dependências
Instale, se necessário: `pip install pdfplumber openpyxl pandas matplotlib numpy scipy`.
O OCR (opcional) precisa de `pytesseract`, `pdf2image` e do binário `poppler` + `tesseract`.
`scipy` só é usado pelo método "Otimização numérica" (opcional: `RODAR_OTIMIZACAO`).

In [ ]:
import io
import re
import datetime as dt
from collections import defaultdict
from copy import copy

import pandas as pd

# Dependências verificadas de forma tolerante para dar mensagens claras.
FALHAS = []
try:
    import pdfplumber
except Exception as e:                      # noqa: BLE001
    FALHAS.append(f"pdfplumber ({e})")
try:
    import openpyxl
    from openpyxl.utils import get_column_letter
except Exception as e:                      # noqa: BLE001
    FALHAS.append(f"openpyxl ({e})")
try:
    import matplotlib
    matplotlib.use("Agg")                   # backend sem tela (gera arquivos)
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyBboxPatch
except Exception as e:                      # noqa: BLE001
    FALHAS.append(f"matplotlib ({e})")

if FALHAS:
    logger.error("Dependências ausentes: %s", "; ".join(FALHAS))
    logger.error("Instale com: pip install pdfplumber openpyxl pandas matplotlib")
else:
    logger.info("Todas as dependências principais disponíveis.")

## Etapa 1 — Extração do PDF

Estratégia (robusta e comentada):

1. **Detectar camada de texto.** O demonstrativo *original da Copel* é um PDF com texto — ideal para
   `pdfplumber`. Se o arquivo vier **escaneado/rasterizado** (só imagens), o texto não existe e o
   código avisa e, se `TENTAR_OCR=True`, tenta OCR. *([Inferência] o comportamento exato depende do
   arquivo emitido pela distribuidora.)*
2. **Ler o cabeçalho** de cada página via *regex* (`UC`, `UC Conces`, `Nome`, tipo Geradora/Beneficiária).
3. **Ler a tabela** por coordenadas (`extract_table`) e mapear os grupos de colunas pelos rótulos do
   cabeçalho, somando as duas sub-colunas de cada métrica com `somar_energia()` — isso neutraliza a
   inversão `PT|FP` ↔ `FP|TP`.
4. Pegar a linha **mais recente** (a primeira linha de dados = mês de referência do arquivo).

### 2.1 Utilidades de parsing

In [ ]:
def limpar_numero(txt) -> float:
    """Converte texto -> float tolerando formatos BR.
    Regra-chave (bug histórico): só tratar '.' como separador de milhar QUANDO há ',' na string.
    Assim '2.32' -> 2.32 (e não 232), mas '1.234,5' -> 1234.5."""
    if txt is None:
        return 0.0
    if isinstance(txt, (int, float)):
        return float(txt)
    s = str(txt).strip().replace("kWh", "").replace("%", "").strip()
    if s in ("", "-", "—"):
        return 0.0
    if "," in s:                       # formato BR com decimal por vírgula
        s = s.replace(".", "").replace(",", ".")
    # se NÃO há vírgula, mantém o ponto como separador decimal (não mexe)
    s = re.sub(r"[^0-9.\-]", "", s)
    try:
        return float(s) if s not in ("", "-", ".", "-.") else 0.0
    except ValueError:
        return 0.0


def somar_energia(a, b) -> float:
    """Soma as duas sub-colunas de uma métrica (PT/FP ou FP/TP).
    Como os valores são mutuamente exclusivos entre sub-colunas, a soma
    recupera o valor correto independentemente da ordem/inversão."""
    return round(limpar_numero(a) + limpar_numero(b), 4)


MESES_BR = {1:"jan",2:"fev",3:"mar",4:"abr",5:"mai",6:"jun",
            7:"jul",8:"ago",9:"set",10:"out",11:"nov",12:"dez"}

def parse_referencia(txt) -> dt.datetime | None:
    """'08/2026' | '8/2026' | '2026-08' -> datetime(2026, 8, 1)."""
    if not txt:
        return None
    m = re.search(r"(\d{1,2})\s*[/\-]\s*(\d{4})", str(txt))
    if m:
        mes, ano = int(m.group(1)), int(m.group(2))
        if 1 <= mes <= 12:
            return dt.datetime(ano, mes, 1)
    m = re.search(r"(\d{4})\s*[/\-]\s*(\d{1,2})", str(txt))   # ISO invertido
    if m:
        ano, mes = int(m.group(1)), int(m.group(2))
        if 1 <= mes <= 12:
            return dt.datetime(ano, mes, 1)
    return None

### 2.2 Camada de texto / OCR opcional

In [ ]:
def pdf_tem_texto(pdf) -> bool:
    """True se ao menos uma das primeiras páginas tiver texto extraível."""
    for page in pdf.pages[:3]:
        if (page.extract_text() or "").strip():
            return True
    return False


def ocr_paginas(pdf_path: Path) -> list[str]:
    """OCR opcional para PDFs escaneados. Retorna o texto por página.
    Requer: pdf2image + poppler e pytesseract + tesseract instalados.
    OBS.: OCR de tabelas numéricas densas é sujeito a erros — trate como
    ponto de partida e valide contra o PDF."""
    try:
        from pdf2image import convert_from_path
        import pytesseract
    except Exception as e:                       # noqa: BLE001
        logger.error("OCR indisponível (%s). Instale pdf2image, pytesseract, poppler e tesseract.", e)
        return []
    textos = []
    for i, img in enumerate(convert_from_path(str(pdf_path), dpi=300), 1):
        textos.append(pytesseract.image_to_string(img, lang="por"))
        logger.info("OCR página %d concluído.", i)
    return textos

### 2.3 Leitura de uma página (cabeçalho + tabela)

Os rótulos de métrica no PDF são: `Saldo Anterior`, `Cred. Receb. Outra UC`, `Energia Injetada`,
`Energia Ativa`, `Credito Utilizado no Mês`, `Percentual`, `Prioridade`, `Saldo Transferido Outra UC`,
`Saldo Final`. Mapeamos **pelos rótulos**, não por posição fixa, para tolerar a inversão de sub-colunas.

In [ ]:
# Rótulos de grupo esperados no topo da tabela (usados para localizar as colunas)
GRUPOS = ["Saldo Anterior", "Cred. Receb", "Energia Injetada", "Energia Ativa",
          "Credito Utilizado", "Percentual", "Prioridade", "Saldo Transferido", "Saldo Final"]


def ler_cabecalho(texto_pagina: str) -> dict:
    """Extrai UC, UC Conces, Nome e o tipo (Geradora/Beneficiária) do cabeçalho."""
    info = {"uc": None, "uc_conces": None, "nome": None, "tipo": None,
            "fase": None, "custo_disp": None}
    m = re.search(r"UC\s*Conces\.?\s*:?\s*(\d+)", texto_pagina, re.I)
    if m:
        info["uc_conces"] = int(m.group(1))                     # demonstrativo antigo
    m = re.search(r"^UC\.?\s*:?\s*(\d{6,})", texto_pagina, re.M)  # "UC. : 000097375703120"
    if m:
        info["uc"] = m.group(1)
        info["uc_conces"] = int(m.group(1))                     # (v4) a UC é a chave
    elif info["uc_conces"] is not None:
        info["uc_conces"] = MIGRACAO_UC_CONCES.get(info["uc_conces"], info["uc_conces"])
    m = re.search(r"Nome\s*:?\s*(.+)", texto_pagina)
    if m:
        info["nome"] = m.group(1).strip()
    m = re.search(r"Tipo da Fase\s*:?\s*([A-Z]{2})\b", texto_pagina)
    info["fase"] = m.group(1) if m else None
    m = re.search(r"Custo de Disp\.?\s*Kwh[ \t]*:?[ \t]*(\d+)", texto_pagina, re.I)
    info["custo_disp"] = int(m.group(1)) if m else None      # kWh/mês não compensáveis
    if re.search(r"UC\s+Benefici", texto_pagina, re.I):
        info["tipo"] = "beneficiaria"
    elif re.search(r"UC\s+Geradora", texto_pagina, re.I):
        info["tipo"] = "geradora"
    return info


def ler_tabela(page) -> list[dict]:
    """Extrai as linhas mensais da tabela de uma página, já somando sub-colunas.
    Retorna lista de dicts com: referencia, consumo, saldo_anterior, cred_receb,
    percentual, saldo_final."""
    tabela = page.extract_table()
    if not tabela:
        return []
    # localizar a linha que contém "Referência" (início do cabeçalho da tabela)
    idx_ref = next((i for i, row in enumerate(tabela)
                    if any(c and "refer" in str(c).lower() for c in row)), 0)
    dados_rows  = tabela[idx_ref + 1:]

    # posição da coluna "Referência" e das duas sub-colunas de cada grupo
    header_flat = [(str(c) if c else "") for c in tabela[idx_ref]]
    col_ref = next((j for j, c in enumerate(header_flat) if "refer" in c.lower()), 0)

    # Para robustez, assumimos o layout padrão do demonstrativo: após "Referência",
    # os grupos vêm em pares de sub-colunas na ordem de GRUPOS (exceto Percentual e
    # Prioridade, que têm 1 coluna). Mapeamos por contagem de pares.
    # Ordem de colunas (0-based, relativo a col_ref):
    #   +1,+2  Saldo Anterior       +3,+4  Cred Receb      +5,+6  Energia Injetada
    #   +7,+8  Energia Ativa        +9,+10 Credito Utiliz  +11    Percentual
    #   +12    Prioridade           +13,+14 Saldo Transf   +15,+16 Saldo Final
    def par(off1, off2):
        return (col_ref + off1, col_ref + off2)
    MAP = {
        "saldo_anterior": par(1, 2), "cred_receb": par(3, 4), "energia_injetada": par(5, 6),
        "energia_ativa": par(7, 8), "credito_util": par(9, 10),
        "percentual": (col_ref + 11,), "saldo_final": par(15, 16),
    }

    linhas = []
    for row in dados_rows:
        if not row or col_ref >= len(row):
            continue
        ref = parse_referencia(row[col_ref])
        if ref is None:
            continue
        def val(cols):
            vs = [row[c] if c < len(row) else 0 for c in cols]
            return somar_energia(*vs) if len(vs) == 2 else limpar_numero(vs[0])
        linhas.append(dict(
            referencia    = ref,
            saldo_anterior= val(MAP["saldo_anterior"]),
            cred_receb    = val(MAP["cred_receb"]),
            energia_ativa = val(MAP["energia_ativa"]),          # -> consumo
            percentual    = val(MAP["percentual"]),
            saldo_final   = val(MAP["saldo_final"]),
        ))
    return linhas

### 2.4 Orquestrador da extração — `extrair_pdf()`

In [ ]:
def extrair_pdf(pdf_path: Path = PDF_PATH, apenas_ultimo_mes: bool = True) -> pd.DataFrame:
    """Percorre o PDF e devolve um DataFrame no schema da aba-mestre `Demonstrativo`.

    apenas_ultimo_mes=True  -> retorna só a linha do mês mais recente de cada UC (uso típico mensal).
    apenas_ultimo_mes=False -> retorna todo o histórico presente no PDF (útil p/ reconstruir a base).
    """
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF não encontrado: {pdf_path.resolve()}")

    registros = []
    with pdfplumber.open(str(pdf_path)) as pdf:
        logger.info("PDF aberto: %d páginas.", len(pdf.pages))
        textos_ocr = None
        if not pdf_tem_texto(pdf):
            logger.warning("PDF SEM camada de texto (provavelmente escaneado/imagem).")
            if TENTAR_OCR:
                logger.info("Tentando OCR…")
                textos_ocr = ocr_paginas(pdf_path)
            if not textos_ocr:
                raise RuntimeError(
                    "Não há texto extraível. Reexporte o demonstrativo como PDF de texto "
                    "(não escaneado) ou habilite o OCR (TENTAR_OCR=True com dependências instaladas).")

        for i, page in enumerate(pdf.pages):
            try:
                texto = (textos_ocr[i] if textos_ocr else page.extract_text()) or ""
                cab = ler_cabecalho(texto)
                linhas = [] if textos_ocr else ler_tabela(page)   # OCR: tabela via texto exigiria parser dedicado
                if not linhas:
                    logger.warning("Página %d: nenhuma linha de tabela lida (UC=%s).",
                                   i + 1, cab.get("uc_conces"))
                    continue
                if apenas_ultimo_mes:
                    linhas = [max(linhas, key=lambda d: d["referencia"])]
                for ln in linhas:
                    registros.append(dict(
                        nome_loja        = None,                     # preenchido depois via mapa uc->loja
                        uc_conces        = cab["uc_conces"],
                        consumo          = ln["energia_ativa"],
                        saldo_anterior_fp= ln["saldo_anterior"],
                        cred_receb_fp    = ln["cred_receb"],
                        percentual       = ln["percentual"],
                        saldo_final_fp   = ln["saldo_final"],
                        arquivo          = pdf_path.name,
                        data_referencia  = ln["referencia"],
                        _tipo            = cab.get("tipo"),
                    ))
            except Exception as e:                                  # noqa: BLE001
                logger.exception("Falha ao processar página %d: %s", i + 1, e)

    df = pd.DataFrame(registros, columns=COLS_MESTRE + ["_tipo"])
    logger.info("Extração concluída: %d registro(s) de %d UC(s).",
                len(df), df["uc_conces"].nunique())
    return df


def ler_cadastro_pdf(pdf_path: Path = PDF_PATH) -> pd.DataFrame:
    """(v2) Lê o CABEÇALHO de todas as páginas: UC Conces, nome, endereço, fase, custo de
    disponibilidade e se a página tem tabela. Serve para o rateio (consumo compensável) e para
    detectar UCs novas sem histórico (páginas sem tabela)."""
    regs = []
    with pdfplumber.open(str(pdf_path)) as pdf:
        for page in pdf.pages:
            texto = page.extract_text() or ""
            cab = ler_cabecalho(texto)
            m = re.search(r"Endere[cç]o\s*:?\s*(.+)", texto)
            tem_tabela = bool(ler_tabela(page))
            # o cabeçalho tem 2 colunas: a linha do Nome/Endereço vem "colada" com Custo de Disp./Etapa
            nome = re.split(r"\s+(?:Valor do Custo|Tipo da Fase|Etapa)\b", cab["nome"] or "")[0] or None
            end = re.split(r"\s+(?:Valor do Custo|Tipo da Fase|Etapa)\b", m.group(1))[0].strip() if m else None
            # sem tabela, o único rótulo da página é o título "UC Geradora / Beneficiária"
            tipo = cab["tipo"] if tem_tabela else "sem_historico"
            regs.append(dict(uc_conces=cab["uc_conces"], nome=nome, tipo=tipo,
                             fase=cab.get("fase"), custo_disp=cab.get("custo_disp"),
                             endereco=end, tem_tabela=tem_tabela))
    df = pd.DataFrame(regs).dropna(subset=["uc_conces"])
    df = (df.groupby("uc_conces", as_index=False)
            .agg(nome=("nome", "first"), tipo=("tipo", lambda t: next((x for x in t if x != "sem_historico"), "sem_historico")), fase=("fase", "first"),
                 custo_disp=("custo_disp", "max"), endereco=("endereco", "first"),
                 tem_tabela=("tem_tabela", "max")))
    logger.info("Cadastro do PDF: %d UC(s); %d sem tabela de histórico.",
                len(df), int((~df["tem_tabela"]).sum()))
    return df

## Etapa 2 — Atualização da planilha (`openpyxl`)

Fluxo:

1. **Carregar** o workbook preservando fórmulas e estilos.
2. Construir o **mapa `UC → nome_loja`**: UCs já gravadas usam o nome do `Demonstrativo`; UCs novas
   usam a aba `Unidades Consumidoras` (v4). Linhas antigas com *UC Conces.* são migradas para a UC nova.
3. **Anexar** as novas linhas em `Demonstrativo` — **idempotente**: se o mês já existir para a UC,
   não duplica.
4. (Opcional) inserir a **linha de fórmula do novo mês** em cada aba de loja, acima de `TOTAL`,
   e estender o intervalo do `TOTAL`.
5. Os resumos se atualizam sozinhos (`P1 = =MAX(...)`).
6. **Salvar** uma cópia. *Fórmulas recém-escritas só ganham valor após um recálculo* — o Excel
   recalcula ao abrir; para pré-calcular sem abrir, use LibreOffice headless.

### 3.1 Carregar workbook e mapa UC → loja

In [ ]:
def carregar_workbook(xlsx_path: Path = XLSX_PATH):
    if not xlsx_path.exists():
        raise FileNotFoundError(f"Planilha não encontrada: {xlsx_path.resolve()}")
    wb = openpyxl.load_workbook(io.BytesIO(Path(xlsx_path).read_bytes()), data_only=False)   # data_only=False => mantém fórmulas
    logger.info("Workbook carregado: %d abas.", len(wb.sheetnames))
    migrar_ucs_demonstrativo(wb)
    return wb


def uc_int(v) -> int | None:
    """'000.097.375.703.120' / '000097375703120' / 97375703120 -> 97375703120 (None se não houver dígitos)."""
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return int(v)
    dig = re.sub(r"\D", "", str(v or ""))
    return int(dig) if dig else None


def migrar_ucs_demonstrativo(wb) -> int:
    """(v4) Converte o UC Conces. antigo das linhas do Demonstrativo para a UC nova."""
    ws = wb[ABA_MESTRE]
    n = 0
    for r in range(2, ws.max_row + 1):
        c = ws.cell(r, COL_IDX["uc_conces"])
        uc = uc_int(c.value)
        if uc is None:
            continue
        if uc in MIGRACAO_UC_CONCES:
            c.value = MIGRACAO_UC_CONCES[uc]
            n += 1
        c.number_format = FMT_UC
    if n:
        logger.info("Demonstrativo: %d linha(s) migrada(s) de UC Conces. para a UC nova.", n)
    return n


def ler_cadastro_ucs(wb) -> dict:
    """{UC:int -> nome} da aba 'Unidades Consumidoras' (UC com ou sem pontos/traço)."""
    if ABA_UCS not in wb.sheetnames:
        logger.warning("Aba %r não encontrada — UCs novas aparecerão como 'UC <nº>'.", ABA_UCS)
        return {}
    cad = {}
    for uc, nome, *_ in wb[ABA_UCS].iter_rows(min_row=2, max_col=2, values_only=True):
        uc = uc_int(uc)
        if uc and nome not in (None, ""):
            cad[uc] = str(nome).strip()
    logger.info("Aba %r: %d UC(s) cadastrada(s).", ABA_UCS, len(cad))
    return cad


def _sem_tag(nome) -> str:
    return re.sub(r"\s*\(?%s\)?\s*$" % TAG_GERADORA, "", str(nome or ""), flags=re.I).strip()


def mapa_uc_para_loja(wb) -> dict:
    """{UC:int -> nome_loja:str}: nome já gravado no Demonstrativo (critério dos SUMIFS das abas de loja)
    e, para UCs que ainda não estão lá, o nome da aba 'Unidades Consumidoras'."""
    ws = wb[ABA_MESTRE]
    mapa = {}
    for r in range(2, ws.max_row + 1):
        loja = ws.cell(r, COL_IDX["nome_loja"]).value
        uc   = uc_int(ws.cell(r, COL_IDX["uc_conces"]).value)
        if loja is not None and uc is not None:
            mapa[uc] = loja
    cad = ler_cadastro_ucs(wb)
    for uc, nome in cad.items():
        if uc not in mapa:
            mapa[uc] = nome
        elif _sem_tag(mapa[uc]).rstrip(".").upper() != _sem_tag(nome).rstrip(".").upper():
            logger.info("UC %015d: nome no Demonstrativo %r (mantido) ≠ cadastro %r.", uc, mapa[uc], nome)
    CADASTRO_UCS.clear(); CADASTRO_UCS.update(cad)
    logger.info("Mapa UC→loja: %d unidades conhecidas.", len(mapa))
    return mapa


CADASTRO_UCS: dict = {}


def conferir_cadastro(df: pd.DataFrame) -> None:
    """Avisa UCs do PDF fora da aba 'Unidades Consumidoras' e UCs Copel do cadastro fora do PDF."""
    if not CADASTRO_UCS or df is None or df.empty:
        return
    pdf = set(int(u) for u in df["uc_conces"].dropna())
    fora = sorted(pdf - set(CADASTRO_UCS))
    if fora:
        logger.warning("UC(s) do demonstrativo fora da aba %r: %s", ABA_UCS,
                       ", ".join(f"{u:015d}" for u in fora))
    # só UCs no formato Copel (15 dígitos com zeros à esquerda: < 10^12) — CNPJ/Cocel ficam de fora
    sem_pdf = [(u, n) for u, n in CADASTRO_UCS.items() if u < 10**12 and u not in pdf]
    for u, n in sorted(sem_pdf, key=lambda x: x[1]):
        logger.info("Cadastro: %s (UC %015d) não aparece no demonstrativo deste mês.", n, u)


def _abas_de_loja(wb) -> list:
    return [ws for ws in wb.worksheets if ws.cell(2, 1).value == "Loja (ref.):"]


def criar_abas_novas_lojas(wb) -> list:
    """(v4) Loja do Demonstrativo sem aba própria: copia a aba de uma loja existente (mesmo layout e
    fórmulas), troca o nome (A1/B2) e acrescenta a loja nos resumos. Os meses são preenchidos depois
    pela reconciliação (`sincronizar_abas_loja`)."""
    if not CRIAR_ABAS_NOVAS_LOJAS:
        return []
    from openpyxl.formula.translate import Translator
    abas = _abas_de_loja(wb)
    if not abas:
        return []
    com_aba = {ws.cell(2, 2).value for ws in abas}
    dem = wb[ABA_MESTRE]
    lojas = sorted({dem.cell(r, 1).value for r in range(2, dem.max_row + 1) if dem.cell(r, 1).value} - com_aba)
    modelo = next((ws for ws in abas if not _eh_geradora(ws.cell(2, 2).value)), abas[0])
    criadas = []
    for loja in lojas:
        ws = wb.copy_worksheet(modelo)
        titulo = re.sub(r"[\[\]\*\?/\\:]", "-", str(loja))[:31]
        ws.title = titulo if titulo not in wb.sheetnames else ws.title
        ws["A1"], ws["B2"] = loja, loja
        wb._sheets.remove(ws)                                    # posição: antes do Demonstrativo
        wb._sheets.insert(wb.sheetnames.index(ABA_MESTRE), ws)
        for nome_res in ABAS_RESUMO:                              # 1 linha nova nos resumos (fórmulas da última loja)
            if nome_res not in wb.sheetnames:
                continue
            wr = wb[nome_res]
            if any(str(wr.cell(r, 1).value or "").strip() == str(loja).strip() for r in range(2, wr.max_row + 1)):
                continue                                         # a loja já está no resumo
            # última linha de loja = última com SUMIFS na coluna B (linhas de total ficam abaixo)
            ult = max((r for r in range(2, wr.max_row + 1)
                       if str(wr.cell(r, 2).value or "").upper().startswith("=SUMIFS")), default=None)
            if ult is None:
                logger.warning("%s: não achei onde acrescentar %r — inclua a linha manualmente.", nome_res, loja)
                continue
            if any(wr.cell(ult + 1, c).value is not None for c in range(1, wr.max_column + 1)):
                wr.insert_rows(ult + 1)                          # abre espaço acima dos totais
            padrao = re.compile(r"(\$?[A-Z]{1,3}\$?)(\d+):(\$?[A-Z]{1,3}\$?)%d(?!\d)" % ult)
            for row in wr.iter_rows():                           # totais SUM(B2:B<ult>) passam a incluir a loja nova
                for cel in row:
                    if isinstance(cel.value, str) and cel.value.startswith("=") and cel.row != ult:
                        cel.value = padrao.sub(lambda m: f"{m.group(1)}{m.group(2)}:{m.group(3)}{ult + 1}"
                                               if int(m.group(2)) <= ult else m.group(0), cel.value)
            for c in range(1, wr.max_column + 1):
                src_c, dst_c = wr.cell(ult, c), wr.cell(ult + 1, c)
                v = src_c.value
                if isinstance(v, str) and v.startswith("="):
                    v = Translator(v, origin=src_c.coordinate).translate_formula(dst_c.coordinate)
                if v is not None or c == 1:
                    dst_c.value = v
                dst_c._style = copy(src_c._style)
            wr.cell(ult + 1, 1, loja)
        logger.info("Loja nova %r: aba %r criada e incluída nos resumos.", loja, ws.title)
        criadas.append(loja)
    return criadas

### 3.2 Anexar linhas na aba-mestre (idempotente)

In [ ]:
def _ultima_linha(ws, col=1) -> int:
    last = 1
    for r in range(1, ws.max_row + 1):
        if ws.cell(r, col).value is not None:
            last = r
    return last


def pares_existentes(ws) -> set:
    """{(uc_conces, 'YYYY-MM')} já presentes na aba-mestre — para não duplicar."""
    existentes = set()
    for r in range(2, ws.max_row + 1):
        uc  = ws.cell(r, COL_IDX["uc_conces"]).value
        ref = ws.cell(r, COL_IDX["data_referencia"]).value
        if uc is not None and isinstance(ref, dt.datetime):
            existentes.add((int(uc), ref.strftime("%Y-%m")))
    return existentes


def anexar_demonstrativo(wb, df: pd.DataFrame, mapa_uc: dict) -> pd.DataFrame:
    """Anexa em `Demonstrativo` as linhas de df que ainda não existem.
    Retorna o DataFrame efetivamente inserido (com nome_loja preenchido).

    Uma mesma UC pode aparecer em 2+ páginas do PDF (ex.: troca de titularidade — um quadro
    zerado e outro com os dados). Todas as páginas da UC são inseridas (os SUMIFS somam as
    linhas); a checagem de duplicidade vale só contra o que já estava na planilha ANTES."""
    ws = wb[ABA_MESTRE]
    ja = pares_existentes(ws)                  # snapshot: não inclui o que for inserido agora
    linha = _ultima_linha(ws, COL_IDX["nome_loja"]) + 1
    inseridas = []
    for _, row in df.iterrows():
        uc = row["uc_conces"]
        if uc is None or pd.isna(uc):
            logger.warning("Registro sem UC ignorado (ref=%s).", row["data_referencia"])
            continue
        uc = int(uc)
        loja = mapa_uc.get(uc, row.get("nome_loja"))
        if loja is None:
            logger.warning("UC %015d fora do Demonstrativo e da aba %r — linha inserida como 'UC %d'. "
                           "Cadastre-a na aba para dar nome à loja.", uc, ABA_UCS, uc)
            loja = f"UC {uc}"
        if row.get("_tipo") == "geradora" and not _eh_geradora(loja):
            loja = f"{loja} ({TAG_GERADORA})"                  # padrão "80 - INDUSTRIA (GERADORA)"
            mapa_uc[uc] = loja
        ref = row["data_referencia"]
        chave = (uc, ref.strftime("%Y-%m"))
        if chave in ja:
            logger.info("Ignorada (já existe): %s / %s", loja, ref.strftime("%m/%Y"))
            continue
        valores = {
            "nome_loja": loja, "uc_conces": uc, "consumo": row["consumo"],
            "saldo_anterior_fp": row["saldo_anterior_fp"], "cred_receb_fp": row["cred_receb_fp"],
            "percentual": row["percentual"], "saldo_final_fp": row["saldo_final_fp"],
            "arquivo": row["arquivo"], "data_referencia": ref,
        }
        for c, v in valores.items():
            cell = ws.cell(linha, COL_IDX[c], v)
            if linha > 2:                                   # mesmo estilo da linha de cima (padrão da planilha)
                cell._style = copy(ws.cell(linha - 1, COL_IDX[c])._style)
            if c == "data_referencia":
                cell.number_format = "mm/yyyy"
            elif c == "uc_conces":
                cell.number_format = FMT_UC
        inseridas.append(valores)
        linha += 1
    logger.info("Aba-mestre: %d nova(s) linha(s) inserida(s).", len(inseridas))
    return pd.DataFrame(inseridas)

### 3.3 Sincronizar as abas de loja com o `Demonstrativo` (reconciliação)

Em vez de só "anexar o mês recém-lido", a função abaixo **reconcilia** cada aba de loja com
a aba-mestre: garante que a aba contenha exatamente os meses presentes no `Demonstrativo`
para aquela loja. Assim ela também **repara divergências** — por exemplo, uma linha de mês
apagada manualmente da aba individual é recriada, mesmo que nenhum PDF novo tenha sido lido.
É idempotente: rodar de novo não duplica nada.

In [ ]:
def _achar_linha_total(ws) -> int | None:
    for r in range(1, ws.max_row + 1):
        v = ws.cell(r, 1).value
        if isinstance(v, str) and v.strip().upper() == "TOTAL":
            return r
    return None


def _sumifs(col_dem: str, ln: int) -> str:
    """Fórmula SUMIFS idêntica ao padrão da planilha, para a coluna `col_dem` do Demonstrativo."""
    return (f'=SUMIFS(Demonstrativo!{col_dem}:{col_dem}, Demonstrativo!A:A, $B$2, '
            f'Demonstrativo!I:I, ">="&$A{ln}, Demonstrativo!I:I, "<="&EOMONTH($A{ln},0))')


PRIMEIRA_LINHA_DADOS = 4   # 1ª linha de dados nas abas de loja (linhas 1-3 = cabeçalho)


def _meses_no_mestre(dem, nome_loja: str) -> list:
    """(ano, mês) distintos presentes na aba-mestre para a loja (casando por nome, col A)."""
    ms = set()
    for r in range(2, dem.max_row + 1):
        if dem.cell(r, 1).value == nome_loja:
            ref = dem.cell(r, COL_IDX["data_referencia"]).value
            if hasattr(ref, "year"):
                ms.add((ref.year, ref.month))
    return sorted(ms)


def sincronizar_abas_loja(wb, apenas_datas: set | None = None) -> list:
    """Reconcilia CADA aba de loja com a aba-mestre `Demonstrativo`.

    Diferentemente de uma inserção gatilhada apenas pelos meses recém-lidos do PDF,
    esta função é **declarativa/idempotente**: ela garante que toda aba de loja
    contenha exatamente as linhas de mês que existem no `Demonstrativo` para aquela
    loja. Portanto ela **repara divergências** — por exemplo, um mês apagado
    manualmente da aba individual é recriado, mesmo sem PDF novo.

    O bloco de meses é reescrito em posições FINAIS (o openpyxl NÃO reajusta
    referências de fórmula ao inserir/remover linhas; reescrever no lugar evita isso).
    `apenas_datas` (opcional) restringe o reparo a esses meses (os demais são mantidos).
    Devolve [(aba, [meses_restaurados])].
    """
    if not ATUALIZAR_ABAS_LOJA:
        return []
    dem = wb[ABA_MESTRE]
    especiais = set(ABAS_RESUMO) | {ABA_MESTRE}
    relatorio = []
    for nome in wb.sheetnames:
        if nome in especiais:
            continue
        ws = wb[nome]
        if ws.cell(2, 1).value != "Loja (ref.):":            # confirma que é aba de loja
            continue
        nome_loja = ws.cell(2, 2).value                       # B2 = critério dos SUMIFS
        meses_alvo = _meses_no_mestre(dem, nome_loja)
        if not meses_alvo:
            continue
        total_row = _achar_linha_total(ws)
        if total_row is None:
            logger.warning("Aba %r: linha TOTAL não encontrada — pulada.", nome)
            continue

        # meses já presentes na aba (para o relatório do que foi restaurado)
        antes = {(v.year, v.month) for v in
                 (ws.cell(r, 1).value for r in range(PRIMEIRA_LINHA_DADOS, total_row))
                 if isinstance(v, dt.datetime)}
        if MANTER_MESES_EXISTENTES:          # mantém os meses já montados na aba (ex.: calendário até dezembro)
            meses_alvo = sorted(set(meses_alvo) | antes)
        if apenas_datas is not None:
            filtro = {(d.year, d.month) for d in apenas_datas}
            meses_alvo = sorted(antes | (set(meses_alvo) & filtro))

        n_alvo = len(meses_alvo)
        # estilo-modelo (1ª linha de dados) para aplicar a linhas novas/uniformizar
        estilo = [copy(ws.cell(PRIMEIRA_LINHA_DADOS, c)._style) for c in range(1, 7)]

        # ajusta a quantidade de linhas físicas do bloco para casar com n_alvo
        linhas_atuais = total_row - PRIMEIRA_LINHA_DADOS
        if n_alvo > linhas_atuais:
            ws.insert_rows(total_row, n_alvo - linhas_atuais)          # abre espaço acima do TOTAL
        elif n_alvo < linhas_atuais:
            ws.delete_rows(PRIMEIRA_LINHA_DADOS, linhas_atuais - n_alvo)
        total_row = PRIMEIRA_LINHA_DADOS + n_alvo                       # TOTAL na nova posição

        # reescreve o bloco em posições finais
        for i, (ano, mes) in enumerate(meses_alvo):
            r = PRIMEIRA_LINHA_DADOS + i
            for c in range(1, 7):
                ws.cell(r, c)._style = copy(estilo[c - 1])
            ws.cell(r, 1, dt.datetime(ano, mes, 1))
            ws.cell(r, 2, _sumifs("C", r))    # Consumo
            ws.cell(r, 3, _sumifs("D", r))    # Saldo Anterior
            ws.cell(r, 4, _sumifs("E", r))    # Energia Injetada (cred_receb_fp)
            ws.cell(r, 5, _sumifs("G", r))    # Saldo Final
            ws.cell(r, 6, f'=IFERROR((D{r}-B{r})/B{r},"")')            # Diferença %

        # TOTAL (SUM cobrindo todo o bloco)
        ult = PRIMEIRA_LINHA_DADOS + n_alvo - 1
        for c, L in [(2, "B"), (3, "C"), (4, "D"), (5, "E")]:
            ws.cell(total_row, c, f"=SUM({L}{PRIMEIRA_LINHA_DADOS}:{L}{ult})")
        ws.cell(total_row, 6, f'=IFERROR((D{total_row}-B{total_row})/B{total_row},"")')

        restaurados = sorted(set(meses_alvo) - antes)
        if restaurados:
            logger.info("Aba %r: %d mês(es) restaurado(s): %s", nome, len(restaurados),
                        ", ".join(f"{m:02d}/{a}" for a, m in restaurados))
            relatorio.append((nome, [f"{m:02d}/{a}" for a, m in restaurados]))
    if not relatorio:
        logger.info("Abas de loja já sincronizadas com o Demonstrativo (nada a reparar).")
    return relatorio


# Compatibilidade retroativa: mantém o nome antigo como um sinônimo da reconciliação.
def atualizar_abas_loja(wb, novas_datas: set | None = None) -> None:
    sincronizar_abas_loja(wb, apenas_datas=novas_datas or None)

### 3.4 Orquestrador da Etapa 2 — `atualizar_excel()`

In [ ]:
def atualizar_excel(df: pd.DataFrame | None = None, xlsx_in: Path = XLSX_PATH,
                    xlsx_out: Path = XLSX_OUT) -> Path:
    """Atualiza a planilha em dois passos:
      1) se houver `df` (dados do PDF), anexa as linhas novas na aba-mestre `Demonstrativo`;
      2) SEMPRE reconcilia as abas de loja com o `Demonstrativo` (`sincronizar_abas_loja`),
         o que também **repara** meses que porventura tenham sido apagados das abas.
    Passar `df=None` executa apenas o passo 2 (modo reparo, sem depender do PDF)."""
    wb = carregar_workbook(xlsx_in)
    novas_datas = set()
    if df is not None and not df.empty:
        mapa = mapa_uc_para_loja(wb)
        conferir_cadastro(df)
        inseridas = anexar_demonstrativo(wb, df, mapa)
        novas_datas = set(inseridas["data_referencia"]) if not inseridas.empty else set()
        criar_abas_novas_lojas(wb)
    else:
        logger.info("Etapa 2 | sem dados novos do PDF — executando apenas a reconciliação (reparo).")

    # reconciliação COMPLETA: garante que cada aba de loja reflita o Demonstrativo
    relatorio = sincronizar_abas_loja(wb)

    wb.save(str(xlsx_out))
    logger.info("Planilha salva em: %s", xlsx_out)
    if novas_datas:
        logger.info("Mês(es) novo(s) inserido(s) no Demonstrativo: %s",
                    ", ".join(sorted(d.strftime("%m/%Y") for d in novas_datas)))
    if relatorio:
        logger.info("Abas de loja reparadas/atualizadas: %d.", len(relatorio))
    logger.info("Obs.: o Excel recalcula as fórmulas ao abrir. Para pré-calcular via terminal: "
                "`libreoffice --headless --convert-to xlsx --outdir <dir> \"%s\"`.", xlsx_out.name)
    return xlsx_out


def reparar_planilha(xlsx_in: Path = XLSX_PATH, xlsx_out: Path = XLSX_OUT) -> Path:
    """Atalho de conveniência: reconcilia as abas de loja com o Demonstrativo SEM ler o PDF.
    Use quando um mês foi apagado de uma aba de loja e você quer recriá-lo."""
    return atualizar_excel(df=None, xlsx_in=xlsx_in, xlsx_out=xlsx_out)

## Etapa 3 — Relatório visual (painel executivo)

A Etapa 3 lê a **aba-mestre `Demonstrativo`** (fonte única de dados brutos), agrega o
**último mês de referência** por loja e produz um painel no estilo executivo com:

- **Faixa de KPIs**: consumo total, energia injetada, cobertura média, nº de lojas em
  superávit e nº de lojas críticas (sempre **excluindo a UC geradora**, que não consome).
- **Tabela por filial**: Consumida, Injetada, Média Injetada (histórica), Saldo, **% Atual**,
  **% Ideal** (com seta de tendência ▲▼), **Inj. p/ Ideal (~kWh)** — a energia estimada que a
  % Ideal produziria no mês, com a variação em kWh em relação à injeção atual — e um *chip* colorido
  de **Status do Saldo**.
- **Rodapé de recomendações**: lojas que precisam de **atenção urgente** (saldo zerado/negativo)
  e lojas com **superávit** acumulado.

> ⚠️ Os KPIs do painel são **do mês de referência**. A cobertura de um mês de inverno (ex.: agosto)
> é muito menor que a cobertura anual, porque a geração solar é sazonal. A decisão do rateio usa o
> **ano inteiro** (Etapa 4).

### Regras de negócio (todas parametrizadas na célula de CONFIG)

| Status | Condição | Cor |
|---|---|---|
| **Superávit** | `saldo > LIM_SUPERAVIT` | verde |
| **Confortável** | `LIM_CONFORTAVEL < saldo ≤ LIM_SUPERAVIT` | azul |
| **Saldo baixo** | `LIM_BAIXO ≤ saldo ≤ LIM_CONFORTAVEL` | âmbar |
| **Crítico / Zero** | `saldo < LIM_BAIXO` | vermelho |

> **% Ideal** — com `ESTRATEGIA_RATEIO = "W"` (padrão) a coluna mostra o % calculado pelo **modelo de
> rateio anual** (Etapa 4): cobertura anual igual para todas as lojas, usando consumo compensável,
> saldo utilizável e sazonalidade da geração. As estratégias legadas continuam disponíveis:
>
> - **"C"**: `necessidade = max(0, (HORIZONTE_MESES + MESES_BUFFER) × consumo_médio − saldo)`; `% ∝ necessidade`.
> - **"A"**: `% = Σpct × consumo_médio ÷ Σ consumo_médio`.
>
> O total de alocação (`Σ percentual` das beneficiárias) é **conservado** e a geradora fica com 0.
> `PISO_PCT` garante uma % mínima por filial.
>
> **De % para kWh** — a conversão usa o fator empírico do próprio mês
> `fator = Σ energia injetada ÷ Σ percentual` (só beneficiárias), então
> `Inj. p/ Ideal ≈ pct_ideal × fator`. É uma **estimativa do mês** (marcada com ~).

### 4.1 Agregação dos dados (último mês por loja)

In [ ]:
import statistics
from collections import defaultdict


def agregar_workbook(xlsx_path: Path = XLSX_OUT):
    """Lê a aba-mestre `Demonstrativo` e devolve (dados, mes_referencia).

    `dados` é uma lista de dicts por loja, referente ao ÚLTIMO mês encontrado:
      loja, consumo, injetada, saldo, pct_atual, media_inj, consumo_medio
    A média injetada usa TODO o histórico da loja; o consumo médio usa os últimos
    `JANELA_MEDIA_MESES` meses (base da sugestão de % para o horizonte de revisão).
    Lê valores brutos (não fórmulas), portanto independe do recálculo do Excel.
    """
    if not Path(xlsx_path).exists():
        # se o arquivo atualizado ainda não existe, cai para o original
        xlsx_path = XLSX_PATH
    logger.info("Etapa 3 | agregando dados de: %s", Path(xlsx_path).name)
    wb = openpyxl.load_workbook(io.BytesIO(Path(xlsx_path).read_bytes()), data_only=False)
    if ABA_MESTRE not in wb.sheetnames:
        raise RuntimeError(f"Aba-mestre '{ABA_MESTRE}' não encontrada em {Path(xlsx_path).name}.")
    ws = wb[ABA_MESTRE]

    c_loja = COL_IDX["nome_loja"]
    c_cons = COL_IDX["consumo"]
    c_cred = COL_IDX["cred_receb_fp"]
    c_pct  = COL_IDX["percentual"]
    c_sald = COL_IDX["saldo_final_fp"]
    c_ref  = COL_IDX["data_referencia"]

    linhas, maxmes = [], None
    for r in range(2, ws.max_row + 1):
        loja = ws.cell(r, c_loja).value
        if loja is None or str(loja).strip() == "":
            continue
        ref = ws.cell(r, c_ref).value
        linhas.append(dict(
            loja=str(loja).strip(),
            consumo=_num(ws.cell(r, c_cons).value),
            cred=_num(ws.cell(r, c_cred).value),
            pct=_num(ws.cell(r, c_pct).value),
            uc=ws.cell(r, COL_IDX["uc_conces"]).value,
            saldo=_num(ws.cell(r, c_sald).value),
            ref=ref,
        ))
        if ref is not None and (maxmes is None or ref > maxmes):
            maxmes = ref

    if not linhas:
        raise RuntimeError("Nenhuma linha de dados encontrada na aba-mestre.")

    # históricos por loja (soma por mês -> média entre meses)
    cons_mes = defaultdict(lambda: defaultdict(float))
    cred_mes = defaultdict(lambda: defaultdict(float))
    for x in linhas:
        cons_mes[x["loja"]][x["ref"]] += x["consumo"]
        cred_mes[x["loja"]][x["ref"]] += x["cred"]
    def _media_recente(meses: dict) -> float:
        """Média dos últimos JANELA_MEDIA_MESES meses (None = todo o histórico)."""
        refs = sorted(r for r in meses if r is not None)
        if JANELA_MEDIA_MESES:
            refs = refs[-int(JANELA_MEDIA_MESES):]
        return statistics.mean(meses[r] for r in refs) if refs else 0.0

    consumo_medio  = {l: _media_recente(m) for l, m in cons_mes.items()}
    media_injetada = {l: statistics.mean(m.values()) for l, m in cred_mes.items()}

    # agregação do último mês (soma linhas duplicadas; a 2ª costuma ser toda-zero)
    ult = {}
    for x in linhas:
        if x["ref"] == maxmes:
            d = ult.setdefault(x["loja"], dict(consumo=0.0, cred=0.0, saldo=0.0, pct=0.0, pct_uc={}))
            d["consumo"] += x["consumo"]
            d["cred"]    += x["cred"]
            d["saldo"]   += x["saldo"]
            # a linha "real" de cada UC carrega o percentual; lojas com 2+ UCs somam as UCs
            d["pct_uc"][x["uc"]] = max(d["pct_uc"].get(x["uc"], 0.0), x["pct"])
            d["pct"] = sum(d["pct_uc"].values())

    dados = []
    for loja in sorted(ult):
        d = ult[loja]
        dados.append(dict(
            loja=loja, consumo=d["consumo"], injetada=d["cred"], saldo=d["saldo"],
            pct_atual=d["pct"], media_inj=media_injetada.get(loja, 0.0),
            consumo_medio=consumo_medio.get(loja, 0.0),
        ))
    logger.info("Etapa 3 | %d lojas agregadas para o mês %s",
                len(dados), maxmes.strftime("%m/%Y") if hasattr(maxmes, "strftime") else maxmes)
    return dados, maxmes


def _num(v) -> float:
    """Converte valor de célula (número ou texto) em float, tolerante a None/vírgula."""
    if v is None:
        return 0.0
    if isinstance(v, (int, float)):
        return float(v)
    try:
        return limpar_numero(v)
    except Exception:
        return 0.0

### 4.2 Classificação de status e sugestão de % ideal

In [ ]:
def classificar(saldo: float):
    """Devolve (rótulo, cor) do status do saldo conforme os limiares de CONFIG."""
    if saldo > LIM_SUPERAVIT:
        return "Superávit", "#1a7f5a"
    if saldo > LIM_CONFORTAVEL:
        return "Confortável", "#2f7fb0"
    if saldo >= LIM_BAIXO:
        return "Saldo baixo", "#c98a1a"
    return "Crítico / Zero", "#b23b3b"


def _eh_geradora(nome: str) -> bool:
    return TAG_GERADORA.upper() in str(nome).upper()


def _aplicar_piso(pesos: dict, total: float, piso: float) -> dict:
    """Reparte `total` proporcionalmente a `pesos`, garantindo `piso` a cada loja.
    Lojas que ficariam abaixo do piso são fixadas nele e o restante é renormalizado
    entre as demais (iterativo). Conserva a soma = total."""
    fixos, livres = {}, dict(pesos)
    while True:
        resto = total - piso * len(fixos)
        soma = sum(livres.values())
        if not livres or resto <= 0:
            base = total / max(len(pesos), 1)
            return {l: base for l in pesos}          # piso inviável: divide igualmente
        parc = {l: (resto * w / soma if soma else resto / len(livres)) for l, w in livres.items()}
        abaixo = [l for l, v in parc.items() if v < piso]
        if not piso or not abaixo:
            return {**{l: piso for l in fixos}, **parc}
        for l in abaixo:
            fixos[l] = piso
            livres.pop(l)


def _arredondar_conservando(valores: dict, casas: int = 2) -> dict:
    """Arredonda para `casas` decimais mantendo a soma (método do maior resto), para que
    a soma das % digitadas no site da Copel seja exatamente o total atual."""
    q = 10 ** casas
    alvo = round(sum(valores.values()) * q)
    base = {k: int(v * q) for k, v in valores.items()}          # trunca
    falta = alvo - sum(base.values())
    for k in sorted(valores, key=lambda k: valores[k] * q - base[k], reverse=True)[:max(falta, 0)]:
        base[k] += 1
    return {k: v / q for k, v in base.items()}


def sugerir_pct_ideal(dados: list, rateio: dict | None = None) -> list:
    """Preenche `pct_ideal`, `inj_ideal_kwh` e `delta_kwh` de cada loja.

    - Estratégia "W" (padrão, v2): usa o % calculado pelo modelo de rateio anual (Etapa 4),
      recebido em `rateio`. Se `rateio` não for informado, cai para a estratégia "C".
    - Estratégias legadas "C" e "A": regra da v1 (ver markdown acima).

    Conversão para ENERGIA (kWh) pelo fator empírico do mês
    `fator = Σ energia injetada / Σ percentual` (beneficiárias):
      - `inj_ideal_kwh` = energia estimada que a % Ideal produziria (≈ pct_ideal × fator);
      - `delta_kwh`     = inj_ideal_kwh − energia injetada atual."""
    global RESUMO_RATEIO
    benef = [d for d in dados if not _eh_geradora(d["loja"])]
    total_pct = sum(d["pct_atual"] for d in benef)
    total_inj = sum(d["injetada"] for d in benef)
    total_cons_med = sum(d["consumo_medio"] for d in benef)
    fator = (total_inj / total_pct) if total_pct else 0.0     # kWh por 1% de alocação

    estrategia = ESTRATEGIA_RATEIO if ESTRATEGIA_RATEIO in ("A", "C", "W") else "W"
    if estrategia == "W" and not rateio:
        logger.warning("% ideal | rateio anual indisponível — usando a estratégia legada C.")
        estrategia = "C"

    if estrategia == "W":
        tab = rateio["tabela"]
        alocado = {d["loja"]: float(tab["pct_ideal"].get(d["loja"], 0.0)) for d in benef}
        if PISO_PCT:
            alocado = _arredondar_conservando(_aplicar_piso(alocado, total_pct, PISO_PCT))
    else:
        pesos = {d["loja"]: d["consumo_medio"] for d in benef}      # estratégia A
        if estrategia == "C":
            meses = HORIZONTE_MESES + MESES_BUFFER
            nec = {d["loja"]: max(0.0, meses * d["consumo_medio"] - d["saldo"]) for d in benef}
            if sum(nec.values()) > 0:
                pesos = nec
            else:
                logger.warning("Estratégia C: nenhuma loja com necessidade no horizonte — usando A.")
                estrategia = "A"
        alocado = _arredondar_conservando(_aplicar_piso(pesos, total_pct, PISO_PCT))

    for d in dados:
        if _eh_geradora(d["loja"]):
            d["pct_ideal"] = 0.0
            d["inj_ideal_kwh"] = 0.0
            d["delta_kwh"] = 0.0
        else:
            d["pct_ideal"] = alocado.get(d["loja"], 0.0)
            d["inj_ideal_kwh"] = round(d["pct_ideal"] * fator)
            d["delta_kwh"] = round(d["inj_ideal_kwh"] - d["injetada"])

    cobertura = (total_inj / total_cons_med) if total_cons_med else 0.0
    RESUMO_RATEIO = dict(estrategia=estrategia, horizonte=HORIZONTE_MESES, buffer=MESES_BUFFER,
                         piso=PISO_PCT, total_pct=total_pct, fator=fator, cobertura=cobertura,
                         soma_ideal=round(sum(d["pct_ideal"] for d in benef), 2))
    if estrategia == "W":
        RESUMO_RATEIO.update(k=rateio["k"], cobertura_anual=rateio["cobertura_estrutural"],
                             inicio=rateio["parametros"]["Início do ciclo"])
    logger.info("%% ideal | estratégia %s, Σpct %.2f → Σideal %.2f, fator ≈ %.1f kWh/%%",
                estrategia, total_pct, RESUMO_RATEIO["soma_ideal"], fator)
    return dados


RESUMO_RATEIO: dict = {}

### 4.3 Renderização do painel (`matplotlib`)

Layout desenhado em coordenadas de eixo (0–1) com altura da figura proporcional ao número
de lojas, de modo que a tabela nunca fique espremida. Salva **PNG** e **PDF**.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import datetime as _dt

# Paleta do painel
COR_FUNDO   = "#f4f6f8"
COR_HEADER  = "#0f3d3e"
COR_HEADER2 = "#155e63"
COR_TXT     = "#1c2b2b"
COR_LINHA_A = "#ffffff"
COR_LINHA_B = "#eef2f3"
COR_KPI     = "#ffffff"


def _fmt(n) -> str:
    """Formata número inteiro com ponto de milhar (padrão pt-BR)."""
    try:
        return f"{float(n):,.0f}".replace(",", ".")
    except Exception:
        return str(n)


def _fmt_sinal(n) -> str:
    """Número com sinal explícito: +57 / −13 / ±0."""
    n = round(float(n))
    return ("+" if n > 0 else ("\u2212" if n < 0 else "\u00b1")) + _fmt(abs(n))


def _wrap(s: str, width: int) -> str:
    """Quebra uma lista separada por vírgulas em várias linhas de largura ~width."""
    out, line = [], ""
    for w in s.split(", "):
        if len(line) + len(w) + 2 > width:
            out.append(line.rstrip(", ")); line = ""
        line += w + ", "
    out.append(line.rstrip(", "))
    return "\n".join([x for x in out if x])


def desenhar_painel(dados: list, maxmes):
    """Desenha o painel executivo e devolve a figura (usada no painel e no relatório PDF)."""
    dados = sorted(dados, key=lambda d: d["loja"])
    n = len(dados)
    row_h  = 0.30
    fig_h  = 5.4 + n * 0.34
    fig = plt.figure(figsize=(13.6, fig_h), dpi=140)
    fig.patch.set_facecolor(COR_FUNDO)
    ax = fig.add_axes([0, 0, 1, 1]); ax.axis("off")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    H = fig_h
    def y(px): return 1 - px / H          # posição vertical em polegadas a partir do topo

    # ----- cabeçalho -----
    ax.add_patch(FancyBboxPatch((0.03, y(1.15)), 0.94, 1.05 / H,
                 boxstyle="round,pad=0.002,rounding_size=0.006",
                 fc=COR_HEADER, ec="none", transform=ax.transAxes, zorder=2))
    ax.text(0.06, y(0.52), "\u2600  Painel de Energia Injetada \u2014 Geração Distribuída",
            fontsize=20, fontweight="bold", color="white", va="center", zorder=3)
    ax.text(0.06, y(0.88), f"{EMPRESA}  \u00b7  {UF}",
            fontsize=11, color="#cfe3e3", va="center", zorder=3)
    ref_txt = f"{MESES_BR.get(maxmes.month, maxmes.month)}/{maxmes.year}" if hasattr(maxmes, "month") else str(maxmes)
    ax.text(0.945, y(0.50), ref_txt.upper(), fontsize=17, fontweight="bold",
            color="white", ha="right", va="center", zorder=3)
    ax.text(0.945, y(0.85), "Mês de referência", fontsize=9, color="#cfe3e3",
            ha="right", va="center", zorder=3)

    # ----- KPIs (excluem a geradora) -----
    benef = [d for d in dados if not _eh_geradora(d["loja"])]
    tot_cons = sum(d["consumo"] for d in benef)
    tot_inj  = sum(d["injetada"] for d in benef)
    cob = (tot_inj / tot_cons * 100) if tot_cons else 0
    n_sup = sum(1 for d in benef if d["status"] == "Superávit")
    n_cri = sum(1 for d in benef if d["status"] == "Crítico / Zero")
    kpis = [
        ("Consumo total",      f"{_fmt(tot_cons)} kWh", COR_HEADER2),
        ("Energia injetada",   f"{_fmt(tot_inj)} kWh",  "#1a7f5a"),
        ("Cobertura média",    f"{cob:.0f}%",           "#2f7fb0"),
        ("Lojas c/ superávit", str(n_sup),              "#1a7f5a"),
        ("Lojas críticas",     str(n_cri),              "#b23b3b"),
    ]
    kx0, kw, kgap = 0.03, 0.182, 0.0075
    ky_top, kh = 1.45, 1.0
    for i, (lab, val, cor) in enumerate(kpis):
        x = kx0 + i * (kw + kgap)
        ax.add_patch(FancyBboxPatch((x, y(ky_top + kh)), kw, kh / H,
                     boxstyle="round,pad=0.002,rounding_size=0.008",
                     fc=COR_KPI, ec="#d9e2e4", lw=1, transform=ax.transAxes, zorder=2))
        ax.add_patch(plt.Rectangle((x, y(ky_top + kh)), 0.006, kh / H, fc=cor, ec="none",
                     transform=ax.transAxes, zorder=3))
        ax.text(x + 0.02, y(ky_top + 0.30), val, fontsize=15.5, fontweight="bold",
                color=cor, va="center", zorder=3)
        ax.text(x + 0.02, y(ky_top + 0.72), lab, fontsize=9, color="#5a6b6b",
                va="center", zorder=3)

    # ----- cabeçalho da tabela -----
    cols = [
        ("Filial / Loja",        0.045, "left"),
        ("Consumida\n(kWh)",     0.40,  "right"),
        ("Injetada\n(kWh)",      0.475, "right"),
        ("Média Inj.\n(~kWh)",   0.55,  "right"),
        ("Saldo\n(kWh)",         0.625, "right"),
        ("% Atual",              0.685, "right"),
        ("% Ideal",              0.745, "right"),
        ("Inj. p/ Ideal\n(~kWh)",0.845, "right"),
        ("Status do Saldo",      0.868, "left"),
    ]
    yh = 2.7
    ax.add_patch(plt.Rectangle((0.03, y(yh + 0.5)), 0.94, 0.5 / H, fc=COR_HEADER2, ec="none",
                 transform=ax.transAxes, zorder=2))
    for name, cx, al in cols:
        ax.text(cx, y(yh + 0.25), name, fontsize=8.4, fontweight="bold", color="white",
                ha=al, va="center", zorder=3, linespacing=0.9)

    # ----- linhas -----
    y0 = yh + 0.5
    for i, d in enumerate(dados):
        yy = y0 + i * row_h
        band = COR_LINHA_A if i % 2 == 0 else COR_LINHA_B
        ax.add_patch(plt.Rectangle((0.03, y(yy + row_h)), 0.94, row_h / H, fc=band, ec="none",
                     transform=ax.transAxes, zorder=1))
        yc = y(yy + row_h / 2)
        ax.text(0.045, yc, d["loja"], fontsize=8.6, color=COR_TXT, va="center", ha="left", zorder=3)
        ax.text(0.40,  yc, _fmt(d["consumo"]),  fontsize=8.6, va="center", ha="right", zorder=3)
        ax.text(0.475, yc, _fmt(d["injetada"]), fontsize=8.6, va="center", ha="right", zorder=3)
        ax.text(0.55,  yc, _fmt(d["media_inj"]),fontsize=8.6, va="center", ha="right", color="#5a6b6b", zorder=3)
        ax.text(0.625, yc, _fmt(d["saldo"]),    fontsize=8.6, va="center", ha="right", zorder=3)
        ax.text(0.685, yc, f'{d["pct_atual"]:.2f}', fontsize=8.6, va="center", ha="right", zorder=3)
        if _eh_geradora(d["loja"]):
            ax.text(0.745, yc, "\u2014", fontsize=8.6, va="center", ha="right", color="#9aa7a7", zorder=3)
            ax.text(0.828, yc, "\u2014", fontsize=8.6, va="center", ha="right", color="#9aa7a7", zorder=3)
        else:
            delta = d["pct_ideal"] - d["pct_atual"]
            seta  = "\u25b2" if delta > 0.05 else ("\u25bc" if delta < -0.05 else "=")
            cseta = "#1a7f5a" if delta > 0.05 else ("#b23b3b" if delta < -0.05 else "#5a6b6b")
            ax.text(0.745, yc, f'{d["pct_ideal"]:.2f}', fontsize=8.6, fontweight="bold",
                    va="center", ha="right", zorder=3)
            ax.text(0.751, yc, seta, fontsize=6.6, va="center", ha="left", color=cseta, zorder=3)
            # energia que a % Ideal produziria + variação (kWh) em relação à atual
            ax.text(0.828, yc, _fmt(d["inj_ideal_kwh"]), fontsize=8.6, fontweight="bold",
                    va="center", ha="right", zorder=3)
            dk = d["delta_kwh"]
            cdk = "#1a7f5a" if dk > 0 else ("#b23b3b" if dk < 0 else "#5a6b6b")
            sinal = "+" if dk > 0 else ("\u2212" if dk < 0 else "\u00b1")
            ax.text(0.862, yc, f'{sinal}{_fmt(abs(dk))}', fontsize=6.4, va="center", ha="right",
                    color=cdk, zorder=3)
        ax.add_patch(FancyBboxPatch((0.868, y(yy + row_h - 0.055)), 0.098, (row_h - 0.11) / H,
                     boxstyle="round,pad=0.001,rounding_size=0.010",
                     fc=d["cor"], ec="none", transform=ax.transAxes, zorder=3))
        ax.text(0.917, yc, d["status"], fontsize=7.1, fontweight="bold", color="white",
                va="center", ha="center", zorder=4)

    # ----- rodapé: recomendações -----
    fy = y0 + n * row_h + 0.35
    criticas = sorted([d for d in dados if d["status"] == "Crítico / Zero" and not _eh_geradora(d["loja"])],
                      key=lambda d: d["consumo"], reverse=True)
    superavit = sorted([d for d in dados if d["status"] == "Superávit" and not _eh_geradora(d["loja"])],
                       key=lambda d: d["saldo"], reverse=True)
    box_h = 2.15
    # caixa: atenção urgente
    ax.add_patch(FancyBboxPatch((0.03, y(fy + box_h)), 0.455, box_h / H,
                 boxstyle="round,pad=0.003,rounding_size=0.008",
                 fc="#fdecec", ec="#e6b3b3", lw=1, transform=ax.transAxes, zorder=2))
    ax.text(0.05, y(fy + 0.32), "\u26a0  Atenção urgente \u2014 reforçar alocação",
            fontsize=11, fontweight="bold", color="#b23b3b", va="center", zorder=3)
    nomes_c = ", ".join(f'{d["loja"].split(" - ")[-1]} ({_fmt_sinal(d["delta_kwh"])})' for d in criticas[:10])
    if len(criticas) > 10:
        nomes_c += f"  (+{len(criticas) - 10} lojas)"
    ax.text(0.05, y(fy + 0.78), _wrap(nomes_c, 60) or "\u2014", fontsize=8.4, color="#7a2e2e",
            va="top", zorder=3, linespacing=1.35)
    ganho_tot = sum(d["delta_kwh"] for d in criticas)
    cap_c = (f"{len(criticas)} loja(s) com saldo zerado/negativo.\n"
             f"Variação total com a % Ideal: {_fmt_sinal(ganho_tot)} kWh/mês (valor por loja entre parênteses).")
    ax.text(0.05, y(fy + box_h - 0.18), cap_c,
            fontsize=7.4, color="#7a2e2e", style="italic", va="bottom", zorder=3,
            linespacing=1.3)
    # caixa: superávit
    ax.add_patch(FancyBboxPatch((0.515, y(fy + box_h)), 0.455, box_h / H,
                 boxstyle="round,pad=0.003,rounding_size=0.008",
                 fc="#e9f6ef", ec="#a9d9c1", lw=1, transform=ax.transAxes, zorder=2))
    ax.text(0.535, y(fy + 0.32), "\u2714  Superávit acumulado \u2014 reduzir alocação",
            fontsize=11, fontweight="bold", color="#1a7f5a", va="center", zorder=3)
    nomes_s = ", ".join(f'{d["loja"].split(" - ")[-1]} ({_fmt_sinal(d["delta_kwh"])})' for d in superavit[:8])
    ax.text(0.535, y(fy + 0.78), _wrap(nomes_s, 58) or "\u2014", fontsize=8.4, color="#1c5c44",
            va="top", zorder=3, linespacing=1.35)
    cap_s = ("Créditos podem expirar (ReN ANEEL).\nReduzir a % destas lojas (% Ideal \u25bc) "
             "para consumir a sobra até a próxima revisão.")
    ax.text(0.535, y(fy + box_h - 0.18), cap_s,
            fontsize=7.4, color="#1c5c44", style="italic", va="bottom", zorder=3,
            linespacing=1.3)

    # ----- nota de rodapé -----
    hoje = _dt.date.today().strftime("%d/%m/%Y")
    ax.text(0.03, y(fy + box_h + 0.35),
            f"Gerado automaticamente em {hoje}  \u00b7  Fonte: {PDF_PATH.name}  \u00b7  "
            f"Regras: Superávit > {_fmt(LIM_SUPERAVIT)} kWh \u00b7 Confortável > {_fmt(LIM_CONFORTAVEL)} \u00b7 "
            f"Baixo \u2265 {LIM_BAIXO} \u00b7 Crítico/Zero < {LIM_BAIXO}  (parâmetros configuráveis)",
            fontsize=7, color="#8a9797", va="center", zorder=3)
    rr = RESUMO_RATEIO or {}
    if rr:
        if rr["estrategia"] == "W":
            regra = (f"estratégia W — cobertura anual equalizada (k = {rr['k'] * 100:.1f}%, "
                     f"ciclo a partir de {rr['inicio']}; detalhes no relatório de rateio)")
            cob_txt = f"Geração anual ≈ {rr['cobertura_anual'] * 100:.0f}% do consumo compensável"
        elif rr["estrategia"] == "C":
            regra = (f"estratégia C — necessidade de {rr['horizonte']} meses "
                     f"(+{rr['buffer']:.1f} de margem) de consumo médio menos o saldo")
            cob_txt = f"A geração do mês cobre ~{rr['cobertura'] * 100:.0f}% do consumo médio"
        else:
            regra = "estratégia A — proporcional ao consumo médio (cobertura uniforme)"
            cob_txt = f"A geração do mês cobre ~{rr['cobertura'] * 100:.0f}% do consumo médio"
        ax.text(0.03, y(fy + box_h + 0.62),
                f"% Ideal: {regra}; Σ% = {rr['total_pct']:.2f}; ~{rr['fator']:.0f} kWh por 1% no mês.  "
                f"{cob_txt}.",
                fontsize=7, color="#8a9797", va="center", zorder=3)

    return fig


def gerar_painel(dados: list, maxmes, saida_base: Path = PAINEL_OUT):
    """Desenha e salva o painel executivo. Devolve (caminho_png, caminho_pdf)."""
    fig = desenhar_painel(dados, maxmes)
    saida_base = Path(saida_base)
    png = saida_base.with_suffix(".png")
    pdf = saida_base.with_suffix(".pdf")
    fig.savefig(png, facecolor=COR_FUNDO, bbox_inches="tight", pad_inches=0.15)
    fig.savefig(pdf, facecolor=COR_FUNDO, bbox_inches="tight", pad_inches=0.15)
    plt.close(fig)
    logger.info("Etapa 3 | painel salvo em: %s  e  %s", png.name, pdf.name)
    return png, pdf


def preparar_dados_painel(xlsx_path: Path = XLSX_OUT, rateio: dict | None = None):
    """Agrega -> % ideal -> classifica. Devolve (dados, maxmes)."""
    dados, maxmes = agregar_workbook(xlsx_path)
    dados = sugerir_pct_ideal(dados, rateio)
    preso = rateio["tabela"]["saldo_preso"] if rateio else {}
    for d in dados:
        d["preso"] = float(preso.get(d["loja"], 0.0)) if rateio else 0.0
        # status pelo saldo UTILIZÁVEL: saldo preso em cadastro antigo não compensa a fatura
        d["status"], d["cor"] = classificar(d["saldo"] - d["preso"])
        if d["preso"] > 0 and d["saldo"] - d["preso"] <= LIM_CONFORTAVEL:
            d["status"], d["cor"] = "Saldo preso", "#6b5b95"
    return dados, maxmes


def gerar_relatorio_visual(xlsx_path: Path = XLSX_OUT, saida_base: Path = PAINEL_OUT,
                           rateio: dict | None = None):
    """Orquestra a Etapa 3: agrega -> % ideal -> classifica -> desenha painel."""
    dados, maxmes = preparar_dados_painel(xlsx_path, rateio)
    png, pdf = gerar_painel(dados, maxmes, saida_base)
    return dados, maxmes, png, pdf

## Etapa 4 — Rateio anual (% fixo por 12 meses)

O % de cada loja só pode ser alterado **uma vez por ano**, então o objetivo não é o % "ótimo hoje", e sim
um % que **aguente os 12 meses** sem deixar saldo parado numa loja enquanto outra paga a fatura.
Este bloco é **o mesmo nos notebooks Copel, CELESC e Nelson**; só o *adaptador* (4.1) muda, porque cada
planilha guarda o `Demonstrativo` num formato diferente.

### Modelo

Com o % fixo `pᵢ` e a energia distribuída no mês `Gₜ`, o saldo de cada loja evolui assim:

```
Sᵢ,ₜ = Sᵢ,ₜ₋₁ + pᵢ·Gₜ − Compᵢ,ₜ        Compᵢ,ₜ = min(Sᵢ,ₜ₋₁ + pᵢ·Gₜ ; Uᵢ,ₜ)
Uᵢ,ₜ = max(0 ; Consumoᵢ,ₜ − CustoDispᵢ − GeraçãoPrópriaᵢ,ₜ)   ← consumo COMPENSÁVEL
Faturadoᵢ,ₜ = Uᵢ,ₜ − Compᵢ,ₜ
```

**Regra de rateio — cobertura equalizada ("water-filling"):**

```
aᵢ = max(0 ; k·Uᵢ,ano + Alvoᵢ − Sᵢ,início)      com k tal que  Σ aᵢ = G_ano
pᵢ = aᵢ / Σ aᵢ   (× Σ% atual, arredondado conservando a soma)
```

`k` é a **cobertura comum**: a maior cobertura que dá para garantir a todas as lojas ao mesmo tempo.
Lojas cujo saldo já passa de `k·U` recebem 0% e consomem o saldo; as demais ficam com a mesma cobertura.

### Tratamentos

| Tema | Como o bloco trata |
|---|---|
| Custo de disponibilidade | informado pelo adaptador (PDF), inferido dos dados (consumo − crédito usado) ou `CUSTO_DISP_PADRAO`; `CUSTO_DISP_MANUAL` sobrepõe tudo |
| Energia distribuível `G` | informada pelo adaptador (ex.: excedentes distribuídos) ou Σ créditos recebidos reescalada para 100% × `FATOR_GERACAO` |
| Mês incompleto | se as beneficiárias ainda não receberam o crédito do último mês, a análise para no último mês completo |
| Previsão de consumo | perfil dos últimos 12 meses × tendência amortecida (`TENDENCIA_*`) |
| Geração própria | UC que também é geradora só precisa do consumo que a própria geração não cobre |
| Saldo preso | linha de cadastro sem consumo com saldo, ou UC cujo saldo não é usado há `MESES_SALDO_PRESO` meses → não conta como saldo |
| Sazonalidade | a simulação começa em `MES_INICIO_CICLO`; o saldo-alvo inclui o **colchão sazonal** exigido por esse mês |
| Meses até a troca | simulados com o **% atual** para estimar o saldo no início do ciclo |
| Sobra estrutural | se geração + saldo passam do consumo, o relatório indica a parcela que poderia ser cedida a outras UCs |

### Validação

- **Comparação de métodos** (atual, legado 12,5×, proporcional, necessidade − saldo, cobertura equalizada,
  otimização numérica) com o % **fixo** sob os `CENARIOS` de estresse.
- **Backtest com origem móvel**: roda automaticamente quando houver **≥ 24 meses** de histórico.

### 4.0 Parâmetros do rateio anual

In [ ]:
# ---- Rateio anual (Etapas 4 e 5) — mesmos parâmetros nos notebooks Copel, CELESC e Nelson
ESTRATEGIA_RATEIO  = "W"      # "W" = cobertura anual equalizada (water-filling) — RECOMENDADO
                              # "C" / "A" = regras legadas da v1 (a % Ideal do painel volta a usá-las)
MES_INICIO_CICLO   = 10       # mês (1-12) em que o novo % passa a valer no site da distribuidora
FATOR_GERACAO      = 1.00     # geração do próximo ciclo ÷ últimos 12 meses (ex.: 0.995 = degradação dos módulos)
TENDENCIA_AMORTECIMENTO = 0.5 # 0 = ignora tendência de consumo; 1 = aplica a tendência inteira
TENDENCIA_LIMITES  = (0.85, 1.25)   # limites do fator de tendência por loja
USAR_BUFFER_SAZONAL = True    # saldo-alvo inclui o colchão sazonal exigido pelo mês de início do ciclo
MARGEM_SEGURANCA   = 0.03     # + fração do consumo anual compensável no saldo-alvo (erro de previsão)
CUSTO_DISP_PADRAO  = 100      # kWh/mês não compensável quando não há outra fonte (TR=100, BI=50, MO=30)
CUSTO_DISP_MANUAL: dict = {}  # {uc: kWh} — sobrepõe PDF / inferência
MESES_SALDO_PRESO  = 3        # nº de meses sem uso do saldo (com consumo) para classificá-lo como "preso"
PESO_SALDO_PRESO   = 0.5      # fração do saldo preso contada como saldo no rateio (só lojas ativas):
                              #   0 = o saldo nunca será liberado  -> a loja recebe % para cobrir tudo sozinha
                              #   1 = a distribuidora vai realocar o saldo antes do ciclo -> a loja recebe menos %
                              #   0.5 = meio-termo: limita o erro nos dois casos (falta vs. sobra de crédito)
AUTO_EXCLUIR_SEM_COMPENSACAO = True  # loja cujo saldo não é usado há MESES_SALDO_PRESO meses recebe 0%
EXCLUIR_DO_RATEIO: list = []  # nomes de loja forçados a 0% (ex.: loja que vai fechar)
RODAR_OTIMIZACAO   = True     # inclui o método "Otimização numérica" na comparação (mais lento)
CENARIOS = {                  # (fator geração, fator consumo) — o % é mantido FIXO em todos
    "Base":                        (1.00, 1.00),
    "Geração −15%":                (0.85, 1.00),
    "Geração +10%":                (1.10, 1.00),
    "Consumo +10%":                (1.00, 1.10),
    "Geração −15% e Consumo +10%": (0.85, 1.10),
}
ABAS_RATEIO = ["Rateio Anual", "Rateio Simulação", "Rateio Métodos", "Rateio Premissas"]
GERAR_ABAS_RATEIO = False     # False = o .xlsx não recebe as abas "Rateio *" (o rateio fica só no relatório PDF)

### 4.1 Adaptador Copel PR — `entrada_rateio_copel()`

Lê a aba-mestre `Demonstrativo` (1 linha por UC × mês; lojas com 2+ UCs somam), o custo de
disponibilidade de cada UC no **cabeçalho do PDF** e as UCs novas (páginas sem histórico).

In [ ]:
def entrada_rateio_copel(xlsx_path: Path = XLSX_OUT, pdf_path: Path = PDF_PATH) -> dict:
    """Converte a planilha PR para o formato padrão do bloco de rateio."""
    cadastro = None
    try:
        cadastro = ler_cadastro_pdf(pdf_path)
    except Exception as e:                                         # noqa: BLE001
        logger.warning("Etapa 4 | cadastro do PDF indisponível (%s) — custo de disponibilidade = padrão.", e)
    xlsx_path = Path(xlsx_path) if Path(xlsx_path).exists() else XLSX_PATH
    df = pd.read_excel(xlsx_path, sheet_name=ABA_MESTRE, usecols=list(range(len(COLS_MESTRE))))
    df.columns = COLS_MESTRE
    df = df.dropna(subset=["nome_loja", "data_referencia", "uc_conces"]).copy()
    std = pd.DataFrame({
        "loja": df["nome_loja"].astype(str).str.strip(),
        "uc": df["uc_conces"].astype("int64"),
        "ref": pd.to_datetime(df["data_referencia"]).dt.to_period("M"),
        "consumo": df["consumo"].map(_num), "saldo_ant": df["saldo_anterior_fp"].map(_num),
        "cred": df["cred_receb_fp"].map(_num), "pct": df["percentual"].map(_num),
        "saldo_fin": df["saldo_final_fp"].map(_num),
        "geradora": df["nome_loja"].map(_eh_geradora),
    })
    cd, novas = {}, []
    if cadastro is not None and not cadastro.empty:
        cd = cadastro.dropna(subset=["custo_disp"]).set_index("uc_conces")["custo_disp"].to_dict()
        conhecidas = set(std["uc"])
        for _, r in cadastro[(cadastro["tipo"] == "sem_historico") & ~cadastro["uc_conces"].isin(conhecidas)].iterrows():
            novas.append(dict(uc=int(r["uc_conces"]), endereco=r["endereco"]))
    return dict(df=std, cd=cd, novas=novas, sistema="Copel · PR (lojas)",
                fonte=f"{Path(pdf_path).name} + aba {ABA_MESTRE}", portal="site da Copel")

### 4.2 Base de dados do rateio (formato padrão, vindo do adaptador)

O adaptador entrega um `DataFrame` com **uma linha por UC × seção × mês** e as colunas
`loja, uc, ref, consumo, saldo_ant, cred, pct, saldo_fin, geradora` (+ opcionais: `G` = energia
distribuível por mês, `cd` = custo de disponibilidade por UC, `geracao_propria`, `novas`, `alertas`).

In [ ]:
import io as _io
import numpy as np
import datetime as dt
from pathlib import Path
try:
    from scipy.optimize import minimize
except Exception:                                   # noqa: BLE001
    minimize = None

MESES_ABREV = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]


def _rotulo_mes(m: int) -> str:
    return MESES_ABREV[(m - 1) % 12]


def _r_fmt(n) -> str:
    try:
        return f"{float(n):,.0f}".replace(",", ".")
    except (TypeError, ValueError):
        return str(n)


def _r_fmt_sinal(n) -> str:
    n = round(float(n))
    return ("+" if n > 0 else ("−" if n < 0 else "±")) + _r_fmt(abs(n))


def _r_arred(valores: dict, casas: int = 2) -> dict:
    """Arredonda mantendo a soma (maior resto) — a soma digitada no site fecha exata."""
    q = 10 ** casas
    alvo = round(sum(valores.values()) * q)
    base = {k: int(v * q) for k, v in valores.items()}
    falta = alvo - sum(base.values())
    for k in sorted(valores, key=lambda k: valores[k] * q - base[k], reverse=True)[:max(falta, 0)]:
        base[k] += 1
    return {k: v / q for k, v in base.items()}


def _inferir_cd(por_uc: pd.DataFrame) -> dict:
    """Custo de disponibilidade pelos dados: mediana de (consumo − crédito usado) nos meses em que sobrou
    saldo (o consumo foi compensado até o limite), aproximada para 0/30/50/100 kWh."""
    ok = por_uc[(por_uc["saldo"] > 0) & (por_uc["consumo"] > 0) & (por_uc["usado"] > 0)]
    res = {}
    for uc, g in ok.groupby("uc"):
        v = float((g["consumo"] - g["usado"]).clip(lower=0).median())
        res[uc] = float(min([0, 30, 50, 100], key=lambda x: abs(x - v)))
    return res


def montar_base_rateio(entrada: dict) -> dict:
    """Séries mensais por loja a partir do formato padrão do adaptador."""
    df = entrada["df"].copy()
    df["ref"] = pd.PeriodIndex(df["ref"], freq="M")
    df["uc"] = df["uc"].astype("int64")
    for c in ["consumo", "saldo_ant", "cred", "pct", "saldo_fin"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
    ger = df["geradora"].astype(bool)
    b = df[~ger].copy()
    meses_all = pd.period_range(b["ref"].min(), b["ref"].max(), freq="M")
    alertas = list(entrada.get("alertas") or [])

    # ---- energia distribuível e último mês COMPLETO (crédito já lançado nas beneficiárias)
    G_dist = b.groupby("ref")["cred"].sum().reindex(meses_all).fillna(0.0)
    G_ext = entrada.get("G")
    ult = meses_all[-1]
    if G_ext is not None:
        G_ext = pd.Series(G_ext).copy()
        G_ext.index = pd.PeriodIndex(G_ext.index, freq="M")
        ult_g = G_ext.index.max()                               # último mês com dado da geradora
        G_ext = G_ext.groupby(level=0).sum().reindex(meses_all).fillna(0.0)
        # mês além do último da geradora (só parte das beneficiárias já tem o mês) não conta como completo
        completos = [m for m in meses_all
                     if m <= ult_g and (G_ext[m] <= 0 or G_dist[m] >= 0.5 * G_ext[m])]
        ult = max(completos) if completos else ult
        if ult < meses_all[-1]:
            alertas.append(dict(tipo="Mês incompleto", loja="—", kwh=float(G_ext[meses_all[-1]]),
                                detalhe=f"de {ult + 1} a {meses_all[-1]} o crédito da geradora ainda não aparece "
                                        f"em todas as beneficiárias — a análise usa até {ult}"))
    meses = pd.period_range(meses_all[0], ult, freq="M")
    b = b[b["ref"] <= ult]
    df = df[df["ref"] <= ult]

    if "usado" in b.columns and b["usado"].notna().any():          # informado pelo adaptador (ex.: Celesc)
        b["usado"] = pd.to_numeric(b["usado"], errors="coerce").fillna(0.0).clip(lower=0)
    else:                                                          # balanço: anterior + recebido − final
        b["usado"] = (b["saldo_ant"] + b["cred"] - b["saldo_fin"]).clip(lower=0)
    por_uc = (b.groupby(["loja", "uc", "ref"], as_index=False)
                .agg(consumo=("consumo", "sum"), usado=("usado", "sum"),
                     saldo=("saldo_fin", "sum"), cred=("cred", "sum")))

    # ---- custo de disponibilidade: manual > adaptador > inferido > padrão
    cd_ent = dict(entrada.get("cd") or {})
    cd_inf = _inferir_cd(por_uc) if entrada.get("inferir_cd") else {}
    origem_cd = {}
    def _cd(uc):
        for fonte, d in (("manual", CUSTO_DISP_MANUAL), ("PDF", cd_ent), ("inferido", cd_inf)):
            if uc in d and d[uc] is not None and not pd.isna(d[uc]):
                origem_cd[uc] = fonte
                return float(d[uc])
        origem_cd[uc] = "padrão"
        return float(CUSTO_DISP_PADRAO)
    por_uc["cd"] = por_uc["uc"].map(_cd)

    # ---- geração própria (UC que também é geradora em outro demonstrativo)
    proprio = entrada.get("geracao_propria") or {}
    por_uc["proprio"] = 0.0
    for uc, serie in proprio.items():
        s = pd.Series(serie).copy(); s.index = pd.PeriodIndex(s.index, freq="M")
        m = por_uc["uc"] == uc
        por_uc.loc[m, "proprio"] = por_uc.loc[m, "ref"].map(s.groupby(level=0).sum()).fillna(0.0).values
    por_uc["U"] = (por_uc["consumo"] - por_uc["cd"] - por_uc["proprio"]).clip(lower=0)

    def _pivot(frame, col):
        return (frame.pivot_table(index="loja", columns="ref", values=col, aggfunc="sum")
                     .reindex(columns=meses).fillna(0.0))
    C, U, S_mes = _pivot(por_uc, "consumo"), _pivot(por_uc, "U"), _pivot(por_uc, "saldo")

    # ---- energia distribuível a 100% das beneficiárias
    pct_mes = b.groupby(["ref", "uc"])["pct"].max().groupby("ref").sum().reindex(meses)
    if G_ext is not None:
        G = G_ext.reindex(meses).fillna(0.0)
    else:
        G = (G_dist.reindex(meses) * 100 / pct_mes.where(pct_mes > 0)).fillna(G_dist.reindex(meses))

    # ---- saldo preso (a) linha de cadastro sem consumo e com saldo no último mês
    lb = b[b["ref"] == ult]
    preso_a = (lb[(lb["consumo"] <= 0) & (lb["saldo_fin"] > 0) & (lb["cred"] <= 0)]
               .groupby("loja")["saldo_fin"].sum())
    # ---- (b) UC com consumo compensável e saldo, mas sem uso nos últimos N meses
    jan = por_uc[por_uc["ref"].isin(meses[-MESES_SALDO_PRESO:])]
    agg = jan.groupby(["loja", "uc"]).agg(U=("U", "sum"), usado=("usado", "sum"))
    s_ult = por_uc[por_uc["ref"] == ult].set_index(["loja", "uc"])["saldo"].reindex(agg.index).fillna(0.0)
    cond = (agg["U"] > 0) & (agg["usado"] <= 0.01 * agg["U"]) & (s_ult > 0)
    preso_b = s_ult[cond].groupby(level=0).sum()
    sem_comp = sorted(preso_b.index)

    lojas = list(C.index)
    S_total = S_mes[ult].reindex(lojas).fillna(0.0)
    preso = pd.concat([preso_a, preso_b], axis=1).max(axis=1).reindex(lojas).fillna(0.0)
    preso = np.minimum(preso, S_total)
    S_util = (S_total - preso).clip(lower=0)
    pct_atual = (lb.groupby(["loja", "uc"])["pct"].max()          # 1 linha "real" por UC
                   .groupby(level=0).sum().reindex(lojas).fillna(0.0))  # loja com 2+ UCs soma
    ucs = b.groupby("loja")["uc"].unique().map(lambda a: ", ".join(map(str, sorted(a))))

    # ---- alertas
    for l in lojas:
        if preso[l] > 0:
            motivo = (f"saldo não é usado há ≥ {MESES_SALDO_PRESO} meses apesar do consumo"
                      if l in sem_comp else "saldo em cadastro antigo/inativo (sem consumo)")
            alertas.append(dict(tipo="Saldo preso", loja=l, kwh=float(preso[l]),
                                detalhe=motivo + " — solicitar realocação/regularização à distribuidora"))
    for l in sem_comp:
        alertas.append(dict(tipo="Sem compensação", loja=l, kwh=float(C.loc[l].iloc[-MESES_SALDO_PRESO:].sum()),
                            detalhe="consumo não está sendo compensado; % ideal = 0 até regularizar"
                                    if AUTO_EXCLUIR_SEM_COMPENSACAO else "consumo não está sendo compensado"))
    ult12 = C.iloc[:, -12:]
    for l in lojas:
        n_obs = int((ult12.loc[l] > 0).sum())
        if 0 < n_obs < min(12, ult12.shape[1]):
            alertas.append(dict(tipo="Histórico incompleto", loja=l, kwh=None,
                                detalhe=f"só {n_obs} de 12 meses com consumo — meses faltantes estimados pela média"))
    pct_loja_mes = b.groupby(["loja", "ref"])["pct"].max().unstack().reindex(index=lojas, columns=meses).fillna(0)
    cred_loja_mes = b.groupby(["loja", "ref"])["cred"].sum().unstack().reindex(index=lojas, columns=meses).fillna(0)
    for l in lojas:
        falhas = [m for m in meses[-12:] if pct_loja_mes.loc[l, m] > 0 and G[m] > 0 and cred_loja_mes.loc[l, m] <= 0]
        if len(falhas) >= 3:
            alertas.append(dict(tipo="Crédito não recebido", loja=l, kwh=None,
                                detalhe=f"com % > 0, não recebeu crédito em {len(falhas)} dos últimos 12 meses "
                                        f"({', '.join(str(m) for m in falhas[:6])}{'…' if len(falhas) > 6 else ''}) — conferir com a distribuidora"))
    lojas_por_uc = por_uc.groupby("uc")["loja"].first()
    for uc in proprio:
        if uc in lojas_por_uc.index:
            l = lojas_por_uc[uc]
            cob = float(por_uc.loc[por_uc["uc"] == uc, "proprio"].sum())
            alertas.append(dict(tipo="Geração própria", loja=l, kwh=cob,
                                detalhe="a UC também é geradora; só recebe créditos para o consumo que a própria "
                                        "geração não cobre (kWh = geração própria nos meses analisados)"))
    for n in entrada.get("novas") or []:
        alertas.append(dict(tipo="UC nova sem histórico", loja=f"UC {n['uc']}", kwh=None,
                            detalhe=f"{n.get('endereco') or ''} — fora do rateio; definir % por loja análoga"))
    g_df = df[ger]
    saldo_ger = float(g_df[g_df["ref"] == g_df["ref"].max()]["saldo_fin"].sum()) if not g_df.empty else 0.0
    if saldo_ger > 0:
        alertas.append(dict(tipo="Saldo na geradora", loja="GERADORA", kwh=saldo_ger,
                            detalhe="saldo acumulado na própria geradora não é distribuído às beneficiárias"))

    logger.info("Etapa 4 | base: %d loja(s) × %d meses (%s a %s); saldo utilizável %s kWh; preso %s kWh.",
                len(lojas), len(meses), meses[0], ult, _r_fmt(S_util.sum()), _r_fmt(preso.sum()))
    cd_mensal = por_uc.groupby("loja")["cd"].sum() / por_uc.groupby("loja")["ref"].nunique()
    return dict(meses=meses, ultimo=ult, C=C, U=U, S_mes=S_mes, G=G, G_dist=G_dist, pct_mes=pct_mes,
                S_total=S_total, preso=preso, S_util=S_util, pct_atual=pct_atual, ucs=ucs,
                sem_compensacao=sem_comp, alertas=alertas, saldo_geradora=saldo_ger, cd_mensal=cd_mensal,
                origem_cd=origem_cd)

### 4.3 Previsão, simulação mensal, cobertura equalizada e métodos comparados

In [ ]:
def projetar_consumo(U12: np.ndarray):
    """Previsão híbrida: perfil dos últimos 12 meses × fator de tendência amortecido.
    Meses iniciais sem consumo (loja/cadastro recente) são preenchidos com a média observada.
    g = média dos 6 meses mais recentes ÷ média dos 6 anteriores (proxy de crescimento — hipótese)."""
    arr = np.array(U12, dtype=float)
    for i in range(arr.shape[0]):
        obs = arr[i] > 0
        if 0 < obs.sum() < arr.shape[1]:
            primeiro = int(np.argmax(obs))
            if primeiro > 0:
                arr[i, :primeiro] = arr[i, obs].mean()
    g = arr[:, 6:].mean(1) / np.maximum(arr[:, :6].mean(1), 1.0)
    g = np.where(arr[:, :6].mean(1) > 0, g, 1.0)
    g = np.clip(1 + TENDENCIA_AMORTECIMENTO * (g - 1), *TENDENCIA_LIMITES)
    return arr * g[:, None], g


def simular(p, U, G, S0):
    """Aplica o % fixo mês a mês. Devolve (saldo[lojas×T], compensado, faturado, recebido)."""
    n, T = U.shape
    S = np.zeros((n, T)); comp = np.zeros((n, T)); rec = np.zeros((n, T))
    s = np.array(S0, dtype=float).copy()
    for t in range(T):
        rec[:, t] = p * G[t]
        disp = s + rec[:, t]
        comp[:, t] = np.minimum(disp, U[:, t])
        s = disp - comp[:, t]
        S[:, t] = s
    return S, comp, U - comp, rec


def water_filling(Uano, S0, Gano, alvo=0.0, ativos=None):
    """Maximiza a cobertura mínima: aᵢ = max(0, k·Uᵢ + alvoᵢ − Sᵢ), Σ aᵢ = Gano (busca binária em k)."""
    Uano = np.asarray(Uano, float); S0 = np.asarray(S0, float)
    alvo = np.broadcast_to(np.asarray(alvo, float), Uano.shape)
    ativos = np.ones_like(Uano, bool) if ativos is None else np.asarray(ativos, bool)
    if not ativos.any() or Uano[ativos].sum() <= 0:
        a = np.where(ativos, 1.0, 0.0)
        return (a / a.sum() if a.sum() else a), float("inf")
    lo, hi = 0.0, max(20.0, 2 * (Gano + S0[ativos].sum()) / Uano[ativos].sum() + 1)
    for _ in range(200):
        k = (lo + hi) / 2
        a = np.where(ativos, np.maximum(0, k * Uano + alvo - S0), 0.0)
        lo, hi = (k, hi) if a.sum() < Gano else (lo, k)
    return (a / a.sum() if a.sum() else a), k


def buffer_sazonal(G12, U12tot):
    """Colchão (em meses de consumo) que cada mês de início exige para não faltar crédito com % fixo,
    e o pico de saldo acumulado durante o ciclo. Linha i = i-ésimo mês da janela de 12 meses."""
    G12, U12tot = np.asarray(G12, float), np.asarray(U12tot, float)
    if G12.sum() <= 0 or U12tot.sum() <= 0:
        return np.zeros((12, 2))
    w = G12 / G12.sum(); u = U12tot / U12tot.sum()
    linhas = []
    for i in range(12):
        d = np.cumsum(np.roll(w - u, -i)) * 12
        linhas.append((max(0.0, -d.min()), max(0.0, d.max())))
    return np.array(linhas)


def metricas(p, U, G, S0, ativos):
    """Métricas objetivas de um % fixo aplicado por 12 meses (só lojas ativas)."""
    S, comp, fat, _ = simular(p, U, G, S0)
    if not ativos.any():
        ativos = np.ones_like(ativos, bool)
    U, S, comp, fat = U[ativos], S[ativos], comp[ativos], fat[ativos]
    Uano = np.maximum(U.sum(1), 1)
    cob = np.minimum(comp.sum(1) / Uano, 1.0)
    Sf = S[:, -1]
    minimo = max(0.0, U.sum() - np.sum(G) * p[ativos].sum() - np.asarray(S0)[ativos].sum())
    return {
        "Faturado (kWh)": fat.sum(),
        "Faturado acima do mínimo (kWh)": max(0.0, fat.sum() - minimo),
        "Saldo final total (kWh)": Sf.sum(),
        "Saldo ocioso >1 mês (kWh)": np.maximum(0, Sf - U[:, -1]).sum(),
        "Saldo final máx. (meses)": (Sf / (Uano / 12)).max(),
        "Lojas cobertura <90%": int((cob < 0.90).sum()),
        "Cobertura mín.": cob.min(),
        "Dispersão cobertura (dp)": cob.std(),
        "Pico de saldo total (kWh)": S.sum(0).max(),
    }


def otimizar(U, G, S0, p0, ativos, w_disp=20000.0):
    """Método 6: minimiza faturado + saldo final acima de 1 mês + w·var(cobertura) na simulação mensal."""
    if minimize is None or ativos.sum() < 2:
        return None
    idx = np.where(ativos)[0]
    Ua, S0a, Uano = U[idx], np.asarray(S0)[idx], np.maximum(U[idx].sum(1), 1)

    def custo(z):
        q = np.exp(z - z.max()); q /= q.sum()
        S, comp, fat, _ = simular(q, Ua, G, S0a)
        cob = comp.sum(1) / Uano
        return fat.sum() + np.maximum(0, S[:, -1] - Ua[:, -1]).sum() + w_disp * cob.var() * len(q)

    r = minimize(custo, np.log(np.maximum(p0[idx], 1e-6)), method="Powell",
                 options={"maxiter": 20000, "xtol": 1e-4})
    q = np.exp(r.x - r.x.max()); q /= q.sum()
    p = np.zeros_like(p0); p[idx] = q
    return p


def _normalizar(v, ativos):
    v = np.where(ativos, np.maximum(np.asarray(v, float), 0), 0.0)
    return v / v.sum() if v.sum() else v


def calcular_metodos(U, C12, Gano, S0, S_total, pct_atual, ativos, alvo):
    """Percentuais (frações, soma 1) de cada método comparado."""
    Uano = U.sum(1)
    Cm = C12.mean(1)
    P = {"Atual": _normalizar(pct_atual, np.ones_like(ativos))}
    P["Legado 12,5×"] = _normalizar(np.maximum(0, (HORIZONTE_MESES + MESES_BUFFER) * Cm - S_total), ativos)
    P["Proporcional ao consumo"] = _normalizar(Cm, ativos)
    P["Necessidade − saldo"] = _normalizar(np.maximum(0, Uano - S0), ativos)
    P["Cobertura equalizada (W)"], k = water_filling(Uano, S0, Gano, alvo, ativos)
    return P, k

### 4.4 Orquestrador da Etapa 4 — `calcular_rateio_anual()` (inclui backtest)

In [ ]:
def _ciclo(meses_12: pd.PeriodIndex, mes_inicio: int) -> list:
    """Índices (0-11) das colunas dos últimos 12 meses reordenados para começar em `mes_inicio`."""
    pos = {m.month: i for i, m in enumerate(meses_12)}
    return [pos[(mes_inicio - 1 + i) % 12 + 1] for i in range(12)]


def backtest_origem_movel(base: dict, ativos_nomes: list) -> pd.DataFrame | None:
    """Backtest real: para cada origem com 12 meses antes e 12 depois, calcula o % só com o passado,
    mantém fixo e aplica na geração/consumo REAIS. Requer ≥ 24 meses de histórico."""
    meses = base["meses"]
    if len(meses) < 24:
        logger.warning("Etapa 4 | backtest com origem móvel requer ≥ 24 meses (há %d). "
                       "Usando apenas simulação prospectiva + estresse.", len(meses))
        return None
    lojas = list(base["U"].index)
    ativos = np.array([l in ativos_nomes for l in lojas])
    Uall, Call, Sall = base["U"].to_numpy(float), base["C"].to_numpy(float), base["S_mes"].to_numpy(float)
    Gall = base["G"].to_numpy(float)
    pct = base["pct_atual"].to_numpy(float)
    linhas = []
    for o in range(12, len(meses) - 11):
        Up, _ = projetar_consumo(Uall[:, o - 12:o])
        S0 = Sall[:, o - 1]
        P, _ = calcular_metodos(Up, Call[:, o - 12:o], Gall[o - 12:o].sum(), S0, S0, pct, ativos, 0.0)
        for nome, p in P.items():
            linhas.append(dict(origem=str(meses[o]), metodo=nome,
                               **metricas(p, Uall[:, o:o + 12], Gall[o:o + 12], S0, ativos)))
    bt = pd.DataFrame(linhas)
    logger.info("Etapa 4 | backtest: %d origens × %d métodos.", bt["origem"].nunique(), bt["metodo"].nunique())
    return bt


def calcular_rateio_anual(entrada: dict) -> dict:
    """Etapa 4 completa. `entrada` vem do adaptador do notebook (formato padrão)."""
    base = montar_base_rateio(entrada)
    lojas = list(base["U"].index)
    n = len(lojas)
    meses12 = base["meses"][-12:]
    if len(meses12) < 12:
        raise RuntimeError(f"O rateio anual precisa de 12 meses de histórico (há {len(meses12)}).")

    excl = set(EXCLUIR_DO_RATEIO) | (set(base["sem_compensacao"]) if AUTO_EXCLUIR_SEM_COMPENSACAO else set())
    ativos = np.array([l not in excl for l in lojas])

    # ---- previsão (ordem de calendário dos últimos 12 meses) e alinhamento ao ciclo
    U12 = base["U"][meses12].to_numpy(float)
    C12 = base["C"][meses12].to_numpy(float)
    Uproj_cal, g = projetar_consumo(U12)
    G_cal = base["G"][meses12].to_numpy(float) * FATOR_GERACAO
    med = np.median(G_cal[G_cal > 0]) if (G_cal > 0).any() else 0.0
    for i in np.where(G_cal > 2.2 * med)[0] if med else []:
        viz = (G_cal[(i - 1) % 12] + G_cal[(i + 1) % 12]) / 2
        base["alertas"].append(dict(tipo="Geração atípica", loja="—", kwh=float(G_cal[i]),
                                    detalhe=f"{meses12[i]} tem {G_cal[i] / med:.1f}× a mediana mensal — na projeção "
                                            f"foi substituído pela média dos meses vizinhos ({_r_fmt(viz)} kWh)"))
        G_cal[i] = viz
    ordem = _ciclo(meses12, MES_INICIO_CICLO)
    Uc, Gc = Uproj_cal[:, ordem], G_cal[ordem]
    Gano = Gc.sum()

    pct_atual = base["pct_atual"].to_numpy(float)
    total_pct = pct_atual.sum()
    p_atual = pct_atual / total_pct if total_pct else np.full(n, 1 / n)
    G_benef = Gc * (total_pct / 100.0 if total_pct else 1.0)       # o que efetivamente vai às lojas

    # ---- meses entre o último dado e o início do ciclo: simulados com o % atual
    gap = (MES_INICIO_CICLO - (base["ultimo"].month + 1)) % 12
    S0 = base["S_util"].to_numpy(float)
    preso_cont = np.where(ativos, PESO_SALDO_PRESO * base["preso"].to_numpy(float), 0.0)
    if preso_cont.sum() > 0:                       # parte do saldo preso que se espera recuperar
        S0 = S0 + preso_cont
        logger.info("Etapa 4 | saldo preso contado a %.0f%% no rateio: %s kWh.",
                    PESO_SALDO_PRESO * 100, _r_fmt(preso_cont.sum()))
    if gap:
        og = _ciclo(meses12, base["ultimo"].month % 12 + 1)[:gap]
        S_gap, *_ = simular(p_atual, Uproj_cal[:, og], G_cal[og] * (total_pct / 100.0 if total_pct else 1.0), S0)
        S0 = S_gap[:, -1]
    logger.info("Etapa 4 | %s: ciclo começa em %s; %d mês(es) até a troca simulados com o %% atual.",
                entrada.get("sistema", ""), _rotulo_mes(MES_INICIO_CICLO), gap)

    # ---- saldo-alvo: colchão sazonal do mês de início + margem
    buf = buffer_sazonal(G_cal, Uproj_cal[ativos].sum(0) if ativos.any() else Uproj_cal.sum(0))
    buf_por_mes = {m.month: buf[i] for i, m in enumerate(meses12)}
    meses_buffer = buf_por_mes[MES_INICIO_CICLO][0] if USAR_BUFFER_SAZONAL else 0.0
    Uano = Uc.sum(1)
    alvo = Uano * (meses_buffer / 12 + MARGEM_SEGURANCA)

    # ---- métodos + otimização
    P, k = calcular_metodos(Uc, C12, G_benef.sum(), S0, base["S_total"].to_numpy(float),
                            pct_atual, ativos, alvo)
    if RODAR_OTIMIZACAO:
        p_opt = otimizar(Uc, G_benef, S0, P["Cobertura equalizada (W)"], ativos)
        if p_opt is not None:
            P["Otimização numérica"] = p_opt

    linhas = []
    for cen, (fg, fu) in CENARIOS.items():
        for nome, p in P.items():
            linhas.append(dict(cenario=cen, metodo=nome, **metricas(p, Uc * fu, G_benef * fg, S0, ativos)))
    comp_df = pd.DataFrame(linhas)

    # ---- % ideal final (W), arredondado conservando Σ% atual
    pW = P["Cobertura equalizada (W)"]
    pct_ideal = _r_arred({l: pW[i] * total_pct for i, l in enumerate(lojas)})
    pI = np.array([pct_ideal[l] for l in lojas]) / (total_pct if total_pct else 1.0)
    S_prop, comp_prop, fat_prop, rec_prop = simular(pI, Uc, G_benef, S0)
    S_atu, comp_atu, _, _ = simular(p_atual, Uc, G_benef, S0)

    # ---- sobra estrutural: geração + saldo acima do consumo (o saldo vai crescer)
    U_at = Uano[ativos].sum() if ativos.any() else 0.0
    sobra = G_benef.sum() + S0[ativos].sum() - U_at - alvo[ativos].sum()
    necessidade = np.where(ativos, np.maximum(0, np.minimum(k, 1.0) * Uano + alvo - S0), 0.0)
    cedivel = max(0.0, 1 - necessidade.sum() / G_benef.sum()) if G_benef.sum() else 0.0
    if sobra > 0.05 * max(U_at, 1):
        base["alertas"].append(dict(
            tipo="Excedente estrutural", loja="—", kwh=float(sobra),
            detalhe=f"geração do ciclo + saldo superam o consumo compensável — o saldo vai crescer. "
                    f"O saldo atual já cobre ~{S0[ativos].sum() / max(U_at / 12, 1):.0f} meses de consumo; "
                    f"≈ {cedivel * 100:.0f}% da energia do ciclo poderia ser rateada para outras UCs"))

    cd_m = base["cd_mensal"].reindex(lojas).fillna(CUSTO_DISP_PADRAO).to_numpy(float)
    obs = []
    for i, l in enumerate(lojas):
        o = []
        if l in excl:
            o.append("fora do rateio (sem compensação)" if l in base["sem_compensacao"] else "excluída (config)")
        if base["preso"][l] > 0:
            o.append(f"{_r_fmt(base['preso'][l])} kWh presos"
                     + (f" ({PESO_SALDO_PRESO:.0%} contado no rateio)" if ativos[i] and PESO_SALDO_PRESO else ""))
        if g[i] >= 1.10:
            o.append(f"consumo crescendo (+{(g[i] - 1) * 100:.0f}%)")
        if g[i] <= 0.92:
            o.append(f"consumo caindo ({(g[i] - 1) * 100:.0f}%)")
        if ativos[i] and Uano[i] <= 0:
            o.append("consumo coberto pela geração própria")
        elif ativos[i] and pW[i] == 0:
            o.append("saldo cobre o ciclo")
        obs.append("; ".join(o))
    tabela = pd.DataFrame({
        "ucs": base["ucs"].reindex(lojas).values,
        "consumo_anual_proj": Uc.sum(1) + cd_m * 12,
        "custo_disp_anual": cd_m * 12,
        "compensavel_proj": Uano,
        "saldo_atual": base["S_total"].values,
        "saldo_preso": base["preso"].values,
        "saldo_inicio": S0,
        "saldo_alvo": np.where(ativos, alvo, 0.0),
        "necessidade": np.where(ativos, np.maximum(0, k * Uano + alvo - S0) if np.isfinite(k) else 0.0, 0.0),
        "pct_atual": pct_atual,
        "pct_ideal": [pct_ideal[l] for l in lojas],
        "kwh_ideal_ano": pI * G_benef.sum(),
        "kwh_atual_ano": p_atual * G_benef.sum(),
        "saldo_min_proj": S_prop.min(1),
        "saldo_final_proj": S_prop[:, -1],
        "cobertura_proj": comp_prop.sum(1) / np.maximum(Uano, 1),
        "cobertura_atual": comp_atu.sum(1) / np.maximum(Uano, 1),
        "tendencia": g,
        "ativo": ativos,
        "observacao": obs,
    }, index=pd.Index(lojas, name="loja"))

    parametros = {
        "Sistema": entrada.get("sistema", ""),
        "Fonte": entrada.get("fonte", ""),
        "Estratégia": "Cobertura anual equalizada (water-filling)",
        "Início do ciclo": _rotulo_mes(MES_INICIO_CICLO),
        "Base histórica": f"{base['meses'][0]} a {base['ultimo']} ({len(base['meses'])} meses)",
        "Energia distribuível no ciclo (kWh)": Gano,
        "Σ % atual das beneficiárias": total_pct,
        "Energia às lojas no ciclo (kWh)": G_benef.sum(),
        "Consumo compensável projetado (kWh)": U_at,
        "Saldo utilizável no início (kWh)": S0[ativos].sum() if ativos.any() else 0.0,
        "Saldo preso (kWh)": base["preso"].sum(),
        "Cobertura comum k": k if np.isfinite(k) else 0.0,
        "Cobertura estrutural (G ÷ U)": G_benef.sum() / max(U_at, 1),
        "Parcela cedível a outras UCs": cedivel,
        "Colchão sazonal (meses)": meses_buffer,
        "Margem de segurança": MARGEM_SEGURANCA,
        "Peso do saldo preso": PESO_SALDO_PRESO,
        "Fator de geração": FATOR_GERACAO,
        "Amortecimento da tendência": TENDENCIA_AMORTECIMENTO,
        "Custo de disponibilidade": ", ".join(sorted({f"{v}" for v in base["origem_cd"].values()})),
    }
    bt = backtest_origem_movel(base, [l for l, a in zip(lojas, ativos) if a])
    logger.info("Etapa 4 | %s: k = %.1f%% | cobertura estrutural %.1f%% | Σ%% ideal = %.2f",
                entrada.get("sistema", ""), parametros["Cobertura comum k"] * 100,
                parametros["Cobertura estrutural (G ÷ U)"] * 100, sum(pct_ideal.values()))
    return dict(tabela=tabela, metodos=P, comparacao=comp_df, k=parametros["Cobertura comum k"],
                cobertura_estrutural=parametros["Cobertura estrutural (G ÷ U)"],
                parametros=parametros, alertas=base["alertas"], backtest=bt,
                meses_ciclo=[_rotulo_mes(MES_INICIO_CICLO + i) for i in range(12)],
                U_ciclo=Uc, G_ciclo=G_benef, ativos=ativos,
                S_prop=S_prop, S_atual=S_atu, rec_prop=rec_prop, fat_prop=fat_prop,
                buffer={m: buf_por_mes[m] for m in range(1, 13) if m in buf_por_mes},
                base=base, portal=entrada.get("portal", "site da distribuidora"))

## Etapa 5 — Saídas: relatório PDF por demonstrativo (e, opcionalmente, abas `Rateio *` no `.xlsx`)

**Excel**: por padrão (`GERAR_ABAS_RATEIO = False`) a planilha **não** recebe abas de rateio — se existirem de
execuções anteriores, são removidas. Com `GERAR_ABAS_RATEIO = True` as abas abaixo são recriadas a cada execução:

| Aba | Conteúdo |
|---|---|
| `Rateio Anual` | tabela final por loja. `k` (célula `B4`) é editável: **Necessidade**, **% recalculado** e **kWh/ano** são fórmulas. A coluna **% Ideal (digitar no site)** é o valor arredondado que soma exatamente o Σ% atual |
| `Rateio Simulação` | saldo projetado mês a mês de cada loja com o % Ideal e com o % Atual |
| `Rateio Métodos` | métricas de cada método × cenário de estresse, % de cada método e, se houver, o backtest |
| `Rateio Premissas` | parâmetros, premissas, hipóteses e alertas (saldo preso, UCs novas, sobra, etc.) |

**PDF** (um por demonstrativo): 1) painel executivo; 2) rateio proposto; 3) gráficos (sazonalidade,
saldo projetado, colchão por mês de início, cobertura por loja); 4) comparação de métodos; 5) premissas e alertas.

### 5.1 Abas `Rateio *` no Excel

In [ ]:
import openpyxl as _oxl
from openpyxl.styles import Alignment as _Al, Border as _Bo, Font as _Fo, PatternFill as _PF, Side as _Si
from openpyxl.utils import get_column_letter as _col

# mesmo padrão visual das planilhas PR/SC: cabeçalho cinza CCCCCC, texto preto em negrito, aba laranja
_R_HEADER, _R_HEADER2 = "000000", "CCCCCC"
_R_COR_ABA = "FF9900"
_R_FILL_H = _PF("solid", fgColor=_R_HEADER2)
_R_FILL_Z = _PF("solid", fgColor="F2F2F2")
_R_FONT_H = _Fo(bold=True, color="000000")
_R_BORDA = _Bo(bottom=_Si(style="thin", color="D9E2E4"))


def _r_cabecalho(ws, linha, titulos, larguras=None):
    for j, t in enumerate(titulos, 1):
        c = ws.cell(linha, j, t)
        c.fill, c.font = _R_FILL_H, _R_FONT_H
        c.alignment = _Al(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[linha].height = 32
    for j, w in enumerate(larguras or [], 1):
        ws.column_dimensions[_col(j)].width = w


def _r_titulo(ws, texto, sub=""):
    ws["A1"] = texto
    ws["A1"].font = _Fo(bold=True, size=13, color=_R_HEADER)
    ws["A1"].fill = _R_FILL_H
    ws.sheet_properties.tabColor = _R_COR_ABA
    if sub:
        ws["A2"] = sub
        ws["A2"].font = _Fo(italic=True, size=9, color="5A6B6B")


TEXTO_PREMISSAS = {
    "Dados observados": [
        "Consumo, créditos recebidos, % e saldo por UC vêm da aba Demonstrativo (extraída do PDF da distribuidora).",
        "O custo de disponibilidade (TR 100 / BI 50 / MO 30 kWh por mês) nunca é compensado: crédito usado = consumo − custo de disponibilidade.",
        "A geração é sazonal (verão > inverno) e o consumo das lojas costuma ser quase plano.",
    ],
    "Premissas": [
        "O % definido vale por 12 meses a partir do mês de início do ciclo e não é alterado no período.",
        "Todas as UCs pagam tarifa equivalente, então o custo total depende do saldo parado, não de qual UC recebe.",
        "Créditos valem 60 meses; saldo parado não se perde de imediato, mas é dinheiro imobilizado e risco de expirar.",
    ],
    "Hipóteses (validar)": [
        "Geração do próximo ciclo = últimos 12 meses × FATOR_GERACAO.",
        "Consumo do próximo ciclo = perfil dos últimos 12 meses × tendência amortecida por loja.",
        "O saldo preso será tratado com a distribuidora; até lá, não conta como saldo utilizável.",
    ],
    "Como usar": [
        "Digite no site da distribuidora o '% Ideal' da página 'Rateio anual proposto' deste relatório (soma = Σ % atual).",
        "UCs com 0% têm saldo suficiente para o ciclo, geração própria ou estão fora do rateio (ver Observação).",
        "Reavalie antes da próxima troca; com ≥ 24 meses de histórico o backtest roda automaticamente.",
    ],
}


def escrever_abas_rateio(rateio: dict, xlsx_path: Path) -> Path:
    """Recria as abas `Rateio *` na planilha informada."""
    xlsx_path = Path(xlsx_path)
    wb = _oxl.load_workbook(_io.BytesIO(xlsx_path.read_bytes()), data_only=False)
    removidas = [n for n in ABAS_RATEIO if n in wb.sheetnames]
    for nome in removidas:
        del wb[nome]
    if not GERAR_ABAS_RATEIO:                      # rateio só no relatório PDF
        if removidas:
            buf = _io.BytesIO(); wb.save(buf); xlsx_path.write_bytes(buf.getvalue())
            logger.info("Etapa 5 | abas de rateio removidas de %s: %s", xlsx_path.name, ", ".join(removidas))
        return xlsx_path
    tab, par = rateio["tabela"], rateio["parametros"]
    hoje = dt.date.today().strftime("%d/%m/%Y")

    # ------------------------------------------------------------ Rateio Anual
    ws = wb.create_sheet(ABAS_RATEIO[0])
    _r_titulo(ws, f"Rateio anual proposto — {par['Sistema']} — ciclo a partir de {par['Início do ciclo']}",
              f"Gerado em {hoje} · base {par['Base histórica']} · % fixo por 12 meses")
    ws["A4"], ws["B4"] = "Cobertura comum k", rateio["k"]
    ws["A5"], ws["B5"] = "Energia às UCs no ciclo (kWh)", par["Energia às lojas no ciclo (kWh)"]
    ws["A6"], ws["B6"] = "Σ % a distribuir", par["Σ % atual das beneficiárias"]
    ws["B4"].number_format, ws["B5"].number_format, ws["B6"].number_format = "0.0%", "#,##0", "0.00"
    ws["B4"].fill = _PF("solid", fgColor="FFF4D6")
    ws["C4"] = "← editável: altere k (ou use Atingir Meta para Σ Necessidade = B5) e veja as fórmulas"
    ws["C4"].font = _Fo(italic=True, size=9, color="5A6B6B")
    titulos = ["Loja / UC", "UC(s)", "Consumo anual proj. (kWh)", "Custo disp. anual (kWh)",
               "Compensável proj. (kWh)", "Saldo atual (kWh)", "Saldo preso (kWh)",
               "Saldo no início do ciclo (kWh)", "Saldo-alvo (kWh)", "Necessidade (kWh)",
               "% Atual", "% Ideal (digitar no site)", "% Ideal recalc. (fórmula)", "Δ p.p.",
               "kWh ideal / ano", "Saldo mínimo proj. (kWh)", "Saldo final proj. (kWh)",
               "Cobertura proj.", "Cobertura c/ % atual", "Tendência consumo", "Observação"]
    L0 = 8
    _r_cabecalho(ws, L0, titulos, [26, 22, 13, 11, 13, 11, 11, 13, 11, 13, 9, 12, 12, 8,
                                   12, 12, 12, 10, 10, 10, 46])
    tab_ord = tab.sort_values("pct_ideal", ascending=False)
    r1 = L0 + 1
    rN = r1 + len(tab_ord) - 1
    for i, (loja, t) in enumerate(tab_ord.iterrows()):
        r = r1 + i
        nec = f"=IF({'TRUE' if t['ativo'] else 'FALSE'},MAX(0,$B$4*E{r}+I{r}-H{r}),0)"
        vals = [loja, t["ucs"], t["consumo_anual_proj"], t["custo_disp_anual"], t["compensavel_proj"],
                t["saldo_atual"], t["saldo_preso"], t["saldo_inicio"], t["saldo_alvo"], nec,
                t["pct_atual"], t["pct_ideal"], f"=IFERROR(J{r}/SUM($J${r1}:$J${rN})*$B$6,0)",
                f"=L{r}-K{r}", f"=IFERROR(L{r}/$B$6*$B$5,0)", t["saldo_min_proj"], t["saldo_final_proj"],
                t["cobertura_proj"], t["cobertura_atual"], t["tendencia"] - 1, t["observacao"]]
        for j, v in enumerate(vals, 1):
            c = ws.cell(r, j, v.item() if hasattr(v, "item") else v)
            c.border = _R_BORDA
            if i % 2:
                c.fill = _R_FILL_Z
            if 3 <= j <= 10 or j in (15, 16, 17):
                c.number_format = "#,##0"
            elif j in (11, 12, 13, 14):
                c.number_format = "0.00"
            elif j in (18, 19):
                c.number_format = "0.0%"
            elif j == 20:
                c.number_format = "+0%;-0%;0%"
        ws.cell(r, 12).font = _Fo(bold=True)
    rt = rN + 1
    ws.cell(rt, 1, "TOTAL").font = _Fo(bold=True)
    for j in (3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15):
        L = _col(j)
        c = ws.cell(rt, j, f"=SUM({L}{r1}:{L}{rN})")
        c.font = _Fo(bold=True)
        c.number_format = "0.00" if j in (11, 12, 13) else "#,##0"
    ws.cell(rt + 1, 1, "Conferência: Σ % Ideal = Σ % a distribuir")
    ws.cell(rt + 1, 12, f'=IF(ROUND(L{rt}-$B$6,2)=0,"OK","DIFERENÇA")')
    ws.freeze_panes = ws.cell(r1, 2)

    # ------------------------------------------------------------ Rateio Simulação
    ws = wb.create_sheet(ABAS_RATEIO[1])
    _r_titulo(ws, "Saldo projetado mês a mês (kWh) — % fixo por 12 meses",
              "Hipótese: geração e consumo do ciclo repetem o perfil dos últimos 12 meses (com tendência).")
    lojas = list(tab.index)
    meses = rateio["meses_ciclo"]
    linha = 4
    for rotulo, S in [("Com % Ideal", rateio["S_prop"]), ("Com % Atual", rateio["S_atual"])]:
        ws.cell(linha, 1, rotulo).font = _Fo(bold=True, color=_R_HEADER)
        linha += 1
        _r_cabecalho(ws, linha, ["Loja / UC", "Início"] + meses, [26, 10] + [9] * 12)
        for i, l in enumerate(lojas):
            r = linha + 1 + i
            ws.cell(r, 1, l)
            ws.cell(r, 2, float(tab.loc[l, "saldo_inicio"])).number_format = "#,##0"
            for t in range(12):
                c = ws.cell(r, 3 + t, float(S[i, t]))
                c.number_format = "#,##0"
                if S[i, t] < 1 and rateio["U_ciclo"][i, t] > 0:
                    c.fill = _PF("solid", fgColor="FDECEC")
        r = linha + 1 + len(lojas)
        ws.cell(r, 1, "TOTAL").font = _Fo(bold=True)
        for j in range(2, 15):
            L = _col(j)
            c = ws.cell(r, j, f"=SUM({L}{linha + 1}:{L}{r - 1})")
            c.font, c.number_format = _Fo(bold=True), "#,##0"
        linha = r + 3
    ws.cell(linha, 1, "Células em vermelho: mês com saldo zerado (parte do consumo vai para a fatura).").font = \
        _Fo(italic=True, size=9, color="7A2E2E")

    # ------------------------------------------------------------ Rateio Métodos
    ws = wb.create_sheet(ABAS_RATEIO[2])
    _r_titulo(ws, "Comparação de métodos — % fixo por 12 meses sob cenários de estresse",
              "Menor é melhor em faturado, saldo ocioso e dispersão; maior é melhor em cobertura mínima.")
    comp = rateio["comparacao"]
    cols = [c for c in comp.columns if c not in ("cenario", "metodo")]
    _r_cabecalho(ws, 4, ["Cenário", "Método"] + cols, [26, 26] + [14] * len(cols))
    for i, (_, rr) in enumerate(comp.iterrows()):
        r = 5 + i
        ws.cell(r, 1, rr["cenario"]); ws.cell(r, 2, rr["metodo"])
        for j, c in enumerate(cols, 3):
            cel = ws.cell(r, j, float(rr[c]))
            cel.number_format = "0.0%" if "Cobertura" in c or "Dispersão" in c else (
                "0.0" if "meses" in c else "#,##0")
            if rr["metodo"] == "Cobertura equalizada (W)":
                cel.font = _Fo(bold=True)
    r = 5 + len(comp) + 2
    ws.cell(r, 1, "% por método (soma = Σ % a distribuir)").font = _Fo(bold=True, color=_R_HEADER)
    metodos = list(rateio["metodos"])
    _r_cabecalho(ws, r + 1, ["Loja / UC"] + metodos)
    tp = par["Σ % atual das beneficiárias"]
    for i, l in enumerate(lojas):
        ws.cell(r + 2 + i, 1, l)
        for j, m in enumerate(metodos, 2):
            ws.cell(r + 2 + i, j, float(rateio["metodos"][m][i] * tp)).number_format = "0.00"
    bt = rateio.get("backtest")
    r = r + 4 + len(lojas)
    ws.cell(r, 1, "Backtest com origem móvel").font = _Fo(bold=True, color=_R_HEADER)
    if bt is None or bt.empty:
        ws.cell(r + 1, 1, "Indisponível: requer ≥ 24 meses de histórico. Rode de novo quando o histórico completar 24 meses.")
    else:
        _r_cabecalho(ws, r + 1, list(bt.columns))
        for i, (_, rr) in enumerate(bt.iterrows()):
            for j, c in enumerate(bt.columns, 1):
                ws.cell(r + 2 + i, j, rr[c] if isinstance(rr[c], str) else float(rr[c]))

    # ------------------------------------------------------------ Rateio Premissas
    ws = wb.create_sheet(ABAS_RATEIO[3])
    _r_titulo(ws, f"Premissas, parâmetros e alertas do rateio anual — {par['Sistema']}")
    _r_cabecalho(ws, 3, ["Parâmetro", "Valor"], [44, 70])
    for i, (kk, v) in enumerate(par.items()):
        ws.cell(4 + i, 1, kk)
        c = ws.cell(4 + i, 2, v.item() if hasattr(v, "item") else v)
        if isinstance(v, float):
            c.number_format = ("0.0%" if kk.startswith(("Cobertura", "Margem", "Parcela"))
                               else ("0.00" if abs(v) < 50 else "#,##0"))
    r = 5 + len(par)
    for bloco, itens in TEXTO_PREMISSAS.items():
        ws.cell(r, 1, bloco).font = _Fo(bold=True, color=_R_HEADER)
        for it in itens:
            r += 1
            ws.cell(r, 1, "• " + it).alignment = _Al(wrap_text=True, vertical="top")
            ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=2)
            ws.row_dimensions[r].height = 30
        r += 2
    ws.cell(r, 1, "Alertas").font = _Fo(bold=True, color="B23B3B")
    _r_cabecalho(ws, r + 1, ["Tipo / Loja", "Detalhe"])
    for i, a in enumerate(rateio["alertas"]):
        ws.cell(r + 2 + i, 1, f"{a['tipo']} — {a['loja']}")
        kwh = f" ({_r_fmt(a['kwh'])} kWh)" if a.get("kwh") else ""
        ws.cell(r + 2 + i, 2, a["detalhe"] + kwh).alignment = _Al(wrap_text=True)

    buf = _io.BytesIO()
    wb.save(buf)
    xlsx_path.write_bytes(buf.getvalue())
    logger.info("Etapa 5 | abas %s gravadas em %s", ", ".join(ABAS_RATEIO), xlsx_path.name)
    return xlsx_path

### 5.2 Relatório PDF (painel + rateio + gráficos + métodos + premissas)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker
from matplotlib.patches import FancyBboxPatch
from matplotlib.backends.backend_pdf import PdfPages

R_FUNDO, R_HEADER, R_HEADER2, R_TXT = "#f4f6f8", "#0f3d3e", "#155e63", "#1c2b2b"
R_LINHA_A, R_LINHA_B = "#ffffff", "#eef2f3"
COR_PROP, COR_ATUAL = "#155e63", "#c98a1a"      # séries "% Ideal" e "% Atual"
COR_NEUTRA, COR_GRADE = "#9aa7a7", "#d9e2e4"


def _pagina_base(titulo: str, subtitulo: str, altura: float = 11.0):
    fig = plt.figure(figsize=(13.6, altura), dpi=140)
    fig.patch.set_facecolor(R_FUNDO)
    ax = fig.add_axes([0, 0, 1, 1]); ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    H = altura
    ax.add_patch(FancyBboxPatch((0.03, 1 - 1.0 / H), 0.94, 0.85 / H,
                 boxstyle="round,pad=0.002,rounding_size=0.006", fc=R_HEADER, ec="none", zorder=2))
    ax.text(0.05, 1 - 0.43 / H, titulo, fontsize=17, fontweight="bold", color="white", va="center", zorder=3)
    ax.text(0.05, 1 - 0.78 / H, subtitulo, fontsize=9.5, color="#cfe3e3", va="center", zorder=3)
    return fig, ax, H


def _estilo_eixo_r(a):
    for s in ("top", "right"):
        a.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        a.spines[s].set_color(COR_GRADE)
    a.tick_params(colors="#5a6b6b", labelsize=8)
    a.grid(axis="y", color=COR_GRADE, lw=0.6)
    a.set_axisbelow(True)
    a.set_facecolor("white")


def _quebrar(texto: str, largura: int) -> list:
    palavras, linhas, atual = texto.split(), [], ""
    for p in palavras:
        if len(atual) + len(p) + 1 > largura:
            linhas.append(atual); atual = ""
        atual += (" " if atual else "") + p
    return linhas + ([atual] if atual else [])


def pagina_painel_png(png: Path):
    """Página 1 a partir do PNG do painel já gerado pelo notebook."""
    img = plt.imread(str(png))
    h, w = img.shape[:2]
    fig = plt.figure(figsize=(13.6, 13.6 * h / w), dpi=140)
    fig.patch.set_facecolor(R_FUNDO)
    ax = fig.add_axes([0, 0, 1, 1]); ax.axis("off")
    ax.imshow(img)
    return fig


def pagina_tabela_rateio(rateio: dict):
    tab = rateio["tabela"].sort_values("pct_ideal", ascending=False)
    par = rateio["parametros"]
    n = len(tab)
    row_h = 0.26
    H = 3.4 + max(n, 4) * row_h
    fig, ax, H = _pagina_base(
        f"Rateio anual proposto — {par['Sistema']} — ciclo a partir de {par['Início do ciclo']}",
        f"Cobertura comum k = {rateio['k'] * 100:.1f}%  ·  energia às UCs no ciclo ≈ "
        f"{_r_fmt(par['Energia às lojas no ciclo (kWh)'])} kWh  ·  Σ% = {par['Σ % atual das beneficiárias']:.2f}"
        f"  ·  % fixo por 12 meses", H)
    def y(px): return 1 - px / H
    X = dict(comp=0.30, util=0.365, preso=0.43, atual=0.485, ideal=0.545, dk=0.62, smin=0.685,
             sfin=0.75, cob=0.835, obs=0.845)
    cols = [("Loja / UC", 0.045, "left"), ("Compensável\nproj. (kWh/ano)", X["comp"], "right"),
            ("Saldo útil\n(kWh)", X["util"], "right"), ("Saldo preso\n(kWh)", X["preso"], "right"),
            ("% Atual", X["atual"], "right"), ("% Ideal", X["ideal"], "right"), ("Δ kWh\n/ano", X["dk"], "right"),
            ("Saldo mín.\nproj.", X["smin"], "right"), ("Saldo final\nproj.", X["sfin"], "right"),
            ("Cobertura\natual → proj.", X["cob"], "right"), ("Observação", X["obs"], "left")]
    yh = 1.25
    ax.add_patch(plt.Rectangle((0.03, y(yh + 0.5)), 0.94, 0.5 / H, fc=R_HEADER2, ec="none", zorder=2))
    for nome, cx, al in cols:
        ax.text(cx, y(yh + 0.25), nome, fontsize=7.8, fontweight="bold", color="white",
                ha=al, va="center", zorder=3, linespacing=0.9)
    y0 = yh + 0.5
    for i, (loja, t) in enumerate(tab.iterrows()):
        yy = y0 + i * row_h
        ax.add_patch(plt.Rectangle((0.03, y(yy + row_h)), 0.94, row_h / H,
                     fc=R_LINHA_A if i % 2 == 0 else R_LINHA_B, ec="none", zorder=1))
        yc = y(yy + row_h / 2)
        kw = dict(fontsize=8, va="center", zorder=3, color=R_TXT if t["ativo"] else "#9aa7a7")
        ax.text(0.045, yc, str(loja)[:40], ha="left", **kw)
        ax.text(X["comp"], yc, _r_fmt(t["compensavel_proj"]), ha="right", **kw)
        ax.text(X["util"], yc, _r_fmt(t["saldo_inicio"]), ha="right", **kw)
        ax.text(X["preso"], yc, _r_fmt(t["saldo_preso"]) if t["saldo_preso"] else "—", ha="right",
                **{**kw, "color": "#b23b3b" if t["saldo_preso"] else "#9aa7a7"})
        ax.text(X["atual"], yc, f"{t['pct_atual']:.2f}", ha="right", **kw)
        d = t["pct_ideal"] - t["pct_atual"]
        ax.text(X["ideal"], yc, f"{t['pct_ideal']:.2f}", ha="right", fontweight="bold", **kw)
        seta, cs = ("▲", "#1a7f5a") if d > 0.05 else (("▼", "#b23b3b") if d < -0.05 else ("=", "#5a6b6b"))
        ax.text(X["ideal"] + 0.005, yc, seta, fontsize=6.4, ha="left", va="center", color=cs, zorder=3)
        dk = t["kwh_ideal_ano"] - t["kwh_atual_ano"]
        ax.text(X["dk"], yc, _r_fmt_sinal(dk), ha="right",
                **{**kw, "color": "#1a7f5a" if dk > 0 else ("#b23b3b" if dk < 0 else "#5a6b6b")})
        ax.text(X["smin"], yc, _r_fmt(t["saldo_min_proj"]), ha="right", **kw)
        ax.text(X["sfin"], yc, _r_fmt(t["saldo_final_proj"]), ha="right", **kw)
        cob_a, cob_p = min(t["cobertura_atual"], 9.99), min(t["cobertura_proj"], 9.99)
        ax.text(X["cob"], yc, f"{cob_a * 100:.0f}% → {cob_p * 100:.0f}%", ha="right", **kw)
        ax.text(X["obs"], yc, (t["observacao"][:30] + "…") if len(t["observacao"]) > 31 else t["observacao"],
                ha="left", fontsize=6.2, va="center", color="#7a2e2e", zorder=3)
    fy = y0 + max(n, 4) * row_h + 0.3
    ax.text(0.03, y(fy),
            "Compensável = consumo − custo de disponibilidade − geração própria.  Saldo útil = saldo no início do ciclo "
            "sem o saldo preso.  Em cinza: fora do rateio.  Δ kWh/ano = energia recebida no ciclo com % Ideal − com % Atual.",
            fontsize=7, color="#8a9797", va="center")
    return fig


def pagina_graficos(rateio: dict):
    fig, ax0, H = _pagina_base("Por que o % precisa aguentar o ano inteiro",
                               f"{rateio['parametros']['Sistema']} · sazonalidade da geração, trajetória do saldo com % fixo, "
                               "colchão por mês de início e cobertura anual por UC", 11.5)
    gs = fig.add_gridspec(3, 2, left=0.07, right=0.97, top=1 - 1.35 / H, bottom=0.06,
                          hspace=0.55, wspace=0.28, width_ratios=[1.25, 1])
    meses = rateio["meses_ciclo"]; x = np.arange(12)
    at = rateio["ativos"] if rateio["ativos"].any() else np.ones_like(rateio["ativos"])
    a = fig.add_subplot(gs[0, 0]); _estilo_eixo_r(a)
    G = rateio["G_ciclo"]; U = rateio["U_ciclo"][at].sum(0)
    a.plot(x, G / max(G.sum(), 1) * 100, color=COR_PROP, lw=2, marker="o", ms=4, label="Energia distribuída")
    if U.sum() > 0:
        a.plot(x, U / U.sum() * 100, color=COR_ATUAL, lw=2, marker="o", ms=4, label="Consumo compensável")
    a.axhline(100 / 12, color=COR_NEUTRA, lw=0.8, ls="--")
    a.set_xticks(x, meses); a.set_ylabel("% do total anual", fontsize=8, color="#5a6b6b")
    a.set_title("Geração × consumo ao longo do ciclo", fontsize=10, loc="left", color=R_TXT, fontweight="bold")
    a.legend(frameon=False, fontsize=7.5, loc="upper right")
    b = fig.add_subplot(gs[1, 0]); _estilo_eixo_r(b)
    b.plot(x, rateio["S_atual"][at].sum(0), color=COR_ATUAL, lw=2, marker="o", ms=4, label="% Atual")
    b.plot(x, rateio["S_prop"][at].sum(0), color=COR_PROP, lw=2, marker="o", ms=4, label="% Ideal")
    b.set_xticks(x, meses); b.set_ylabel("kWh", fontsize=8, color="#5a6b6b")
    b.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: _r_fmt(v)))
    b.set_title("Saldo total projetado (% fixo)", fontsize=10, loc="left", color=R_TXT, fontweight="bold")
    b.legend(frameon=False, fontsize=7.5)
    c = fig.add_subplot(gs[2, 0]); _estilo_eixo_r(c)
    buf = rateio["buffer"]
    vals = [buf.get(m, (0, 0))[0] for m in range(1, 13)]
    cores = [COR_PROP if m == MES_INICIO_CICLO else "#b9c6c6" for m in range(1, 13)]
    c.bar(np.arange(12), vals, color=cores, width=0.7, edgecolor=R_FUNDO, lw=1)
    c.set_xticks(np.arange(12), [_rotulo_mes(m) for m in range(1, 13)])
    c.set_ylabel("meses de consumo", fontsize=8, color="#5a6b6b")
    c.set_title(f"Colchão de saldo exigido por mês de troca (atual: {_rotulo_mes(MES_INICIO_CICLO)})",
                fontsize=10, loc="left", color=R_TXT, fontweight="bold")
    for i, v in enumerate(vals):
        c.text(i, v + 0.03, f"{v:.1f}", ha="center", fontsize=7, color="#5a6b6b")
    i_esc = MES_INICIO_CICLO - 1
    c.annotate("mês da troca", (i_esc, vals[i_esc]), xytext=(i_esc, max(max(vals), 0.2) * 0.55), ha="center",
               fontsize=7.5, color=COR_PROP, fontweight="bold",
               arrowprops=dict(arrowstyle="-|>", color=COR_PROP, lw=1))
    d = fig.add_subplot(gs[:, 1]); _estilo_eixo_r(d); d.grid(axis="y", visible=False)
    d.grid(axis="x", color=COR_GRADE, lw=0.6)
    tab = rateio["tabela"][rateio["tabela"]["ativo"] & (rateio["tabela"]["compensavel_proj"] > 0)]
    tab = tab.sort_values("cobertura_atual", ascending=True)       # pior cobertura atual no topo
    n_lin = max(len(tab), 8)
    yy = n_lin - 1 - np.arange(len(tab))
    ca = np.minimum(tab["cobertura_atual"].to_numpy() * 100, 100)
    cp = np.minimum(tab["cobertura_proj"].to_numpy() * 100, 100)
    d.hlines(yy, np.minimum(ca, cp), np.maximum(ca, cp), color=COR_GRADE, lw=2)
    d.scatter(ca, yy, color=COR_ATUAL, s=26, zorder=3, label="% Atual", edgecolor="white", lw=0.8)
    d.scatter(cp, yy, color=COR_PROP, s=26, zorder=4, label="% Ideal", edgecolor="white", lw=0.8)
    d.set_yticks(yy, [str(i)[:30] for i in tab.index], fontsize=6.8 if len(tab) > 12 else 8)
    d.set_ylim(-0.6, n_lin - 0.4)
    d.set_xlabel("cobertura do consumo compensável no ciclo (%)", fontsize=8, color="#5a6b6b")
    d.set_title("Cobertura anual por UC", fontsize=10, loc="left", color=R_TXT, fontweight="bold")
    d.legend(frameon=False, fontsize=7.5, loc="lower right")
    return fig


def pagina_metodos(rateio: dict):
    comp = rateio["comparacao"]
    base = comp[comp["cenario"] == "Base"].set_index("metodo")
    pior = comp.groupby("metodo").agg({"Faturado acima do mínimo (kWh)": "max",
                                       "Saldo ocioso >1 mês (kWh)": "max", "Cobertura mín.": "min"})
    metodos = list(base.index)
    fig, ax, H = _pagina_base(f"Comparação de métodos — {rateio['parametros']['Sistema']} — % fixo por 12 meses",
                              "Cenário base (perfil dos últimos 12 meses) e pior caso entre os cenários de estresse: "
                              + ", ".join(CENARIOS), 8.5)
    def y(px): return 1 - px / H
    cols = [("Método", 0.045, "left"), ("Faturado\n(kWh)", 0.31, "right"),
            ("Faturado acima\ndo mínimo", 0.40, "right"), ("Saldo final\n(kWh)", 0.48, "right"),
            ("Saldo ocioso\n>1 mês", 0.56, "right"), ("Cobertura\nmínima", 0.63, "right"),
            ("Dispersão\ncobertura", 0.70, "right"),
            ("PIOR CASO\nfat. acima mín.", 0.80, "right"), ("PIOR CASO\nsaldo ocioso", 0.88, "right"),
            ("PIOR CASO\ncob. mínima", 0.955, "right")]
    yh = 1.3
    ax.add_patch(plt.Rectangle((0.03, y(yh + 0.55)), 0.94, 0.55 / H, fc=R_HEADER2, ec="none", zorder=2))
    for nome, cx, al in cols:
        ax.text(cx, y(yh + 0.27), nome, fontsize=7.8, fontweight="bold", color="white", ha=al, va="center",
                zorder=3, linespacing=0.9)
    row_h = 0.42
    for i, m in enumerate(metodos):
        yy = yh + 0.55 + i * row_h
        destaque = m == "Cobertura equalizada (W)"
        ax.add_patch(plt.Rectangle((0.03, y(yy + row_h)), 0.94, row_h / H,
                     fc="#e9f6ef" if destaque else (R_LINHA_A if i % 2 == 0 else R_LINHA_B), ec="none", zorder=1))
        yc = y(yy + row_h / 2); bb = base.loc[m]; p = pior.loc[m]
        kw = dict(fontsize=8.6, va="center", zorder=3, color=R_TXT, fontweight="bold" if destaque else "normal")
        vals = [m + ("  ← recomendado" if destaque else ""), _r_fmt(bb["Faturado (kWh)"]),
                _r_fmt(bb["Faturado acima do mínimo (kWh)"]), _r_fmt(bb["Saldo final total (kWh)"]),
                _r_fmt(bb["Saldo ocioso >1 mês (kWh)"]), f"{bb['Cobertura mín.'] * 100:.0f}%",
                f"{bb['Dispersão cobertura (dp)'] * 100:.1f} p.p.",
                _r_fmt(p["Faturado acima do mínimo (kWh)"]), _r_fmt(p["Saldo ocioso >1 mês (kWh)"]),
                f"{p['Cobertura mín.'] * 100:.0f}%"]
        for (nome, cx, al), v in zip(cols, vals):
            ax.text(cx, yc, v, ha=al, **kw)
    fy = yh + 0.55 + len(metodos) * row_h + 0.45
    notas = [
        "Faturado acima do mínimo = energia paga na fatura além do déficit inevitável (consumo − geração − saldo). É o que um rateio melhor pode evitar.",
        "Saldo ocioso = saldo final acima de 1 mês de consumo: crédito parado numa UC enquanto outras pagam fatura.",
        "Cobertura mínima / dispersão = justiça entre UCs. Numa mesma empresa o custo total depende do saldo parado, não de qual UC recebe.",
        "Com poucas UCs ou geração sobrando, os métodos ficam próximos; a diferença aparece no saldo ocioso.",
    ]
    bt = rateio.get("backtest")
    notas.append("Backtest com origem móvel: " + ("indisponível (requer ≥ 24 meses de histórico)."
                                                 if bt is None or bt.empty else
                                                 f"{bt['origem'].nunique()} origens avaliadas — ver aba 'Rateio Métodos'."))
    for i, t in enumerate(notas):
        ax.text(0.04, y(fy + i * 0.32), "• " + t, fontsize=8.2, color="#3c4b4b", va="center")
    return fig


def pagina_premissas(rateio: dict):
    fig, ax, H = _pagina_base(f"Premissas, parâmetros e alertas — {rateio['parametros']['Sistema']}",
                              f"Separação entre dado observado, premissa e hipótese — revise antes de aplicar no {rateio['portal']}",
                              11.0)
    def y(px): return 1 - px / H
    yy = 1.35
    ax.text(0.04, y(yy), "Parâmetros do cálculo", fontsize=11, fontweight="bold", color=R_HEADER2)
    yy += 0.3
    for kk, v in rateio["parametros"].items():
        if isinstance(v, float):
            v = (f"{v * 100:.1f}%" if kk.startswith(("Cobertura", "Margem", "Parcela"))
                 else (f"{v:.2f}" if abs(v) < 50 else f"{_r_fmt(v)}"))
        ax.text(0.05, y(yy), kk, fontsize=8.2, color="#5a6b6b", va="center")
        ax.text(0.28, y(yy), str(v)[:48], fontsize=8.2, color=R_TXT, va="center", fontweight="bold")
        yy += 0.21
    yy2 = 1.35
    ax.text(0.52, y(yy2), "Alertas", fontsize=11, fontweight="bold", color="#b23b3b")
    yy2 += 0.3
    if not rateio["alertas"]:
        ax.text(0.53, y(yy2), "Nenhum alerta.", fontsize=8, color="#5a6b6b", va="center")
    for a in rateio["alertas"][:22]:
        kwh = f" ({_r_fmt(a['kwh'])} kWh)" if a.get("kwh") else ""
        txt = f"{a['tipo']} — {a['loja']}{kwh}: {a['detalhe']}"
        for j, linha in enumerate(_quebrar(txt, 92)):
            ax.text(0.53 if j == 0 else 0.545, y(yy2), ("⚠ " if j == 0 else "") + linha,
                    fontsize=7.6, color="#7a2e2e", va="center")
            yy2 += 0.19
        yy2 += 0.05
    yy = max(yy, yy2) + 0.3
    for bloco, itens in TEXTO_PREMISSAS.items():
        ax.text(0.04, y(yy), bloco, fontsize=10, fontweight="bold", color=R_HEADER2)
        yy += 0.27
        for it in itens:
            for j, linha in enumerate(_quebrar(it, 170)):
                ax.text(0.05, y(yy), ("• " if j == 0 else "   ") + linha, fontsize=8, color="#3c4b4b", va="center")
                yy += 0.2
        yy += 0.12
    ax.text(0.03, y(H - 0.25), f"Gerado automaticamente em {dt.date.today().strftime('%d/%m/%Y')}  ·  "
            f"Fonte: {rateio['parametros']['Fonte']}", fontsize=7, color="#8a9797", va="center")
    return fig


def gerar_relatorio_pdf(rateio: dict, painel=None, saida: Path | None = None) -> Path:
    """Relatório completo de UM demonstrativo em um único PDF.
    `painel`: função que devolve a figura do painel, caminho do PNG do painel, ou None."""
    paginas = []
    if callable(painel):
        paginas.append(painel)
    elif painel is not None and Path(painel).exists():
        paginas.append(lambda: pagina_painel_png(Path(painel)))
    paginas += [lambda: pagina_tabela_rateio(rateio), lambda: pagina_graficos(rateio),
                lambda: pagina_metodos(rateio), lambda: pagina_premissas(rateio)]
    with PdfPages(str(saida)) as pdf:
        for fazer in paginas:
            fig = fazer()
            pdf.savefig(fig, facecolor=R_FUNDO)
            plt.close(fig)
        info = pdf.infodict()
        info["Title"] = f"Energia Injetada e Rateio Anual — {rateio['parametros']['Sistema']}"
    logger.info("Etapa 5 | relatório PDF salvo em: %s", Path(saida).name)
    return Path(saida)

## 6. Pipeline completo — `main()`

Executa as cinco etapas em sequência, com tratamento de erros e logs. Cada etapa é
independente: se o PDF não puder ser lido (por exemplo, quando o demonstrativo vier como
imagem escaneada sem OCR disponível), o pipeline **ainda gera o painel** a partir dos dados
já existentes na planilha, registrando o aviso.

In [ ]:
def main(pdf_path: Path = PDF_PATH,
         xlsx_in: Path = XLSX_PATH,
         xlsx_out: Path = XLSX_OUT,
         painel_out: Path = PAINEL_OUT,
         relatorio_out: Path = RELATORIO_OUT,
         apenas_ultimo_mes: bool = False):   # (v4) lê todo o PDF: preenche meses faltantes (idempotente)
    """Pipeline: 1 (PDF -> DataFrame) -> 2 (atualiza Excel) -> 4 (rateio anual) -> 3 (painel) -> 5 (xlsx + PDF)."""
    logger.info("=" * 68)
    logger.info("INÍCIO DO PIPELINE — Energia Injetada (v2)")
    logger.info("=" * 68)

    df = None
    xlsx_para_painel = xlsx_in       # se a Etapa 2 falhar, o painel usa a planilha original

    # -------- Etapa 1: extração do PDF --------
    try:
        df = extrair_pdf(pdf_path, apenas_ultimo_mes=apenas_ultimo_mes)
        if df is None or df.empty:
            logger.warning("Etapa 1 | nenhuma linha extraída do PDF.")
            df = None
        else:
            logger.info("Etapa 1 | %d linha(s) extraída(s) do PDF.", len(df))
    except Exception as e:
        logger.error("Etapa 1 | falha na extração do PDF: %s", e)
        logger.warning("Etapa 1 | seguindo sem novos dados (o painel usará a planilha atual).")
        df = None

    # -------- Etapa 2: atualização do Excel --------
    try:
        xlsx_out = atualizar_excel(df, xlsx_in, xlsx_out)
        xlsx_para_painel = xlsx_out
        logger.info("Etapa 2 | planilha atualizada com sucesso.")
    except Exception as e:
        logger.error("Etapa 2 | falha ao atualizar a planilha: %s", e)
        logger.warning("Etapa 2 | seguindo com a planilha original (%s).", Path(xlsx_in).name)
        xlsx_para_painel = xlsx_in

    # -------- Etapa 4: rateio anual (antes do painel, que usa a % Ideal) --------
    rateio = None
    try:
        rateio = calcular_rateio_anual(entrada_rateio_copel(xlsx_para_painel, pdf_path))
        logger.info("Etapa 4 | rateio anual calculado.")
    except Exception as e:
        logger.exception("Etapa 4 | falha no rateio anual: %s", e)

    # -------- Etapa 3: painel visual --------
    dados = maxmes = png = pdf = None
    try:
        dados, maxmes, png, pdf = gerar_relatorio_visual(xlsx_para_painel, painel_out, rateio)
        logger.info("Etapa 3 | painel gerado com sucesso.")
    except Exception as e:
        logger.exception("Etapa 3 | falha ao gerar o painel: %s", e)

    # -------- Etapa 5: saídas (abas Rateio no xlsx + relatório PDF) --------
    relatorio = None
    if rateio is not None:
        try:
            escrever_abas_rateio(rateio, xlsx_out if Path(xlsx_out).exists() else xlsx_para_painel)
        except Exception as e:
            logger.exception("Etapa 5 | falha ao gravar abas do rateio: %s", e)
        if dados is not None:
            try:
                relatorio = gerar_relatorio_pdf(rateio, lambda: desenhar_painel(dados, maxmes), relatorio_out)
            except Exception as e:
                logger.exception("Etapa 5 | falha ao gerar o relatório PDF: %s", e)

    logger.info("=" * 68)
    logger.info("FIM DO PIPELINE")
    for rot, p in [("Planilha", xlsx_out), ("Painel", png), ("Relatório", relatorio)]:
        if p:
            logger.info("%s: %s", rot, p)
    logger.info("=" * 68)
    return dict(df=df, dados=dados, mes=maxmes, xlsx=xlsx_out, png=png, pdf=pdf,
                relatorio=relatorio, rateio=rateio)


# Execute o pipeline completo:
resultado = main()

### 6.1 Pré-visualização do painel e do rateio no notebook (opcional)

In [ ]:
# Exibe o PNG do painel e a tabela do rateio dentro do próprio notebook, se disponíveis.
try:
    from IPython.display import Image, display
    if resultado.get("png") and Path(resultado["png"]).exists():
        display(Image(filename=str(resultado["png"])))
    if resultado.get("rateio"):
        t = resultado["rateio"]["tabela"]
        display(t[["compensavel_proj", "saldo_inicio", "saldo_preso", "pct_atual", "pct_ideal",
                   "saldo_final_proj", "cobertura_proj", "observacao"]]
                .sort_values("pct_ideal", ascending=False)
                .style.format({"compensavel_proj": "{:,.0f}", "saldo_inicio": "{:,.0f}", "saldo_preso": "{:,.0f}",
                               "pct_atual": "{:.2f}", "pct_ideal": "{:.2f}", "saldo_final_proj": "{:,.0f}",
                               "cobertura_proj": "{:.0%}"}))
except Exception as e:
    logger.warning("Pré-visualização indisponível: %s", e)

## 7. Notas de operação e solução de problemas

**Sobre o formato do demonstrativo.** O código foi escrito para o caso comum em que o
demonstrativo da Copel é um **PDF com camada de texto** — `pdfplumber` extrai as tabelas
diretamente e a leitura é exata. O notebook detecta automaticamente a ausência de texto
(`pdf_tem_texto`) e, se `TENTAR_OCR=True`, aciona o **OCR** (`pdf2image` + `pytesseract`,
que exigem *poppler* e *tesseract* instalados no sistema). O OCR é um plano B: revise os
números extraídos quando ele for usado.

> ⚠️ **Atenção:** se o arquivo recebido for na verdade um **ZIP de imagens** (páginas
> escaneadas) apenas renomeado para `.pdf`, `pdfplumber` falhará com um erro de estrutura.
> Nesse caso, converta as imagens em um PDF real (ou solicite o PDF de texto à Copel) antes
> de rodar a Etapa 1. Enquanto isso, o pipeline continua gerando o painel com os dados já
> presentes na planilha.

**Arquitetura orientada a fórmulas.** A planilha é *fórmula-driven*: as abas de loja e as
abas de resumo (`Resumo Ultimo Mês`, `Resumo Anual (2026)`) usam `SUMIFS` que apontam para a
aba-mestre **`Demonstrativo`**. Por isso, o ponto de inserção de dados novos é **apenas a aba
`Demonstrativo`** — os resumos recalculam sozinhos quando o arquivo é aberto no Excel. A
inserção nas abas individuais (`ATUALIZAR_ABAS_LOJA`) apenas replica as **fórmulas** do mês,
mantendo o padrão visual da planilha.

**Recálculo das fórmulas.** `openpyxl` grava fórmulas **sem** o valor em cache. O Excel/
LibreOffice recalcula ao abrir. Para pré-calcular via terminal (por exemplo, para conferência
automática), use:
```bash
libreoffice --headless --convert-to xlsx --outdir <pasta> "Acompanhamento Energia Injetada PR (atualizado).xlsx"
```

**Idempotência.** `anexar_demonstrativo` não duplica meses já presentes: a chave é
`(uc_conces, AAAA-MM)`. Rodar o notebook duas vezes com o mesmo PDF não gera linhas repetidas.

**Reparo de abas de loja (reconciliação).** As abas individuais são mantidas em sincronia com a
aba-mestre por `sincronizar_abas_loja`, que garante que cada aba contenha exatamente os meses
presentes no `Demonstrativo` para aquela loja. Consequência prática: **se você apagar a linha de
um mês de uma aba de loja, basta rodar o notebook (ou chamar `reparar_planilha()`) que ela é
recriada** — inclusive sem PDF novo, porque o dado bruto continua no `Demonstrativo`. Para reparar
sem executar o pipeline inteiro:
```python
reparar_planilha()          # reconcilia todas as abas de loja e salva em XLSX_OUT
```
> Observação importante: o `Demonstrativo` é a **fonte da verdade**. Reparar uma aba de loja apenas
> recria as *fórmulas* que leem o `Demonstrativo`. Se o mês tiver sido apagado **do próprio
> `Demonstrativo`**, ele só volta relendo o PDF (Etapa 1) — pois é ali que os dados brutos entram.

**Parâmetros que você provavelmente vai ajustar** (célula de CONFIG):
`LIM_SUPERAVIT`, `LIM_CONFORTAVEL`, `LIM_BAIXO` (faixas de status);
`HORIZONTE_MESES`, `CONSIDERAR_SALDO`, `JANELA_MEDIA_MESES`, `MESES_BUFFER`, `PISO_PCT`
(cálculo do **% ideal**);
`DEDUPLICAR_UCS`, `ATUALIZAR_ABAS_LOJA`, `TENTAR_OCR` (comportamento de execução).

**Colunas — esclarecimento importante.** A coluna exibida como *"Energia Injetada"* nas abas
de loja corresponde a **`cred_receb_fp`** (Créditos Recebidos de Outra UC) do demonstrativo —
e **não** à coluna literal "Energia Injetada" do PDF. Esse mapeamento foi validado célula a
célula contra o demonstrativo (ex.: XAXIM, 08/2025, conferência 100%).

**(v2) Rateio anual — o que ajustar e o que conferir.**
- `MES_INICIO_CICLO`: o mês em que o novo % entra em vigor. Muda muito o resultado: começando em set/out o
  colchão sazonal necessário é ~0; começando em abr/mai/jun é ~1,5–1,8 mês de consumo (o saldo alto em
  março/abril é, em boa parte, estoque necessário para o inverno).
- `CUSTO_DISP_MANUAL`: use se o PDF não trouxer o custo de disponibilidade de alguma UC.
- `EXCLUIR_DO_RATEIO` / `AUTO_EXCLUIR_SEM_COMPENSACAO`: lojas fora do rateio recebem 0%.
- Saldo **preso** (cadastro antigo/inativo ou UC que não compensa) é listado nos alertas e **não conta** como
  saldo utilizável — solicite a realocação à Copel (ReN 1.000/2021, art. 655-M) e rode de novo.
- A aba `Rateio Anual` permite editar `k` (célula `B4`) e ver Necessidade / % / kWh recalculados por fórmula.
- Com **≥ 24 meses** na aba `Demonstrativo`, o backtest com origem móvel roda sozinho e aparece na aba
  `Rateio Métodos` e no relatório.